In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import f1_score


# =====================================================================
# 0. 데이터 로드
# =====================================================================

print("=" * 80)
print("데이터 로딩")
print("=" * 80)

train_file = 'train_features.csv'

train_df = pd.read_csv(train_file, low_memory=False)
train_df['time'] = pd.to_datetime(train_df['time'])

print(f"전체 데이터: {train_df.shape}")
print(f"기간: {train_df['time'].min()} ~ {train_df['time'].max()}")
print()


# =====================================================================
# 1. Feature 설정
# =====================================================================

drop_cols = [
    'label',
    'anomaly_type',
    'time',
    'dataset',
    'depth',
    'depth_diff',
    'station',
    'layer'
]

all_features = [
    c for c in train_df.columns
    if c not in drop_cols
]

print(f"사용 피처 개수: {len(all_features)}")
print()


# =====================================================================
# 2. 여러 Validation 시나리오 정의
# =====================================================================

validation_scenarios = {

    # -------------------------------------------------------------
    # 기존 방식
    # Train : 2024 + 2025 H2
    # Valid : 2025 H1
    # -------------------------------------------------------------
    'A_2024+25H2_to_25H1': {
        'train_start': '2024-01-01',
        'train_end':   '2025-12-31 23:59:59',
        'valid_start': '2025-01-01',
        'valid_end':   '2025-06-30 23:59:59',
        'exclude_from_train': [
            ('2025-01-01', '2025-06-30 23:59:59')
        ]
    },

    # -------------------------------------------------------------
    # 2024 -> 2025 H1
    # -------------------------------------------------------------
    'B_2024_to_25H1': {
        'train_start': '2024-01-01',
        'train_end':   '2024-12-31 23:59:59',
        'valid_start': '2025-01-01',
        'valid_end':   '2025-06-30 23:59:59',
        'exclude_from_train': []
    },

    # -------------------------------------------------------------
    # 2024 -> 2025 H2
    # -------------------------------------------------------------
    'C_2024_to_25H2': {
        'train_start': '2024-01-01',
        'train_end':   '2024-12-31 23:59:59',
        'valid_start': '2025-07-01',
        'valid_end':   '2025-12-31 23:59:59',
        'exclude_from_train': []
    },

    # -------------------------------------------------------------
    # 2025 H1 -> 2025 H2
    # -------------------------------------------------------------
    'D_25H1_to_25H2': {
        'train_start': '2025-01-01',
        'train_end':   '2025-06-30 23:59:59',
        'valid_start': '2025-07-01',
        'valid_end':   '2025-12-31 23:59:59',
        'exclude_from_train': []
    },

    # -------------------------------------------------------------
    # 2025 H2 -> 2025 H1
    # -------------------------------------------------------------
    'E_25H2_to_25H1': {
        'train_start': '2025-07-01',
        'train_end':   '2025-12-31 23:59:59',
        'valid_start': '2025-01-01',
        'valid_end':   '2025-06-30 23:59:59',
        'exclude_from_train': []
    },
}


# =====================================================================
# 3. Domain Rule
# =====================================================================

def apply_strict_domain_rules(df):

    df_sorted = df.sort_values('time').copy()

    s = df_sorted['base_pred'].astype(int).copy()

    # -------------------------------------------------------------
    # Rule 1. Flatline
    # -------------------------------------------------------------

    if 'flatline_length' in df_sorted.columns:

        s = pd.Series(
            np.where(
                df_sorted['flatline_length'] >= 12,
                1,
                s.values
            ),
            index=df_sorted.index
        )

    # -------------------------------------------------------------
    # Rule 2. Gap Filling
    # -------------------------------------------------------------

    is_zero = pd.Series(
        s.values == 0,
        index=df_sorted.index
    )

    zero_groups = pd.Series(
        s.values != 0,
        index=df_sorted.index
    ).cumsum()

    zero_len = (
        is_zero
        .groupby(zero_groups)
        .transform('sum')
    )

    first_group = (
        zero_groups.iloc[0]
        if s.iloc[0] == 0
        else -1
    )

    last_group = (
        zero_groups.iloc[-1]
        if s.iloc[-1] == 0
        else -1
    )

    is_internal_zero = (
        is_zero
        & (zero_groups != first_group)
        & (zero_groups != last_group)
    )

    s = pd.Series(
        np.where(
            is_internal_zero & (zero_len <= 17),
            1,
            s.values
        ),
        index=df_sorted.index
    )

    # -------------------------------------------------------------
    # Rule 3. Speckle Removal
    # -------------------------------------------------------------

    is_one = pd.Series(
        s.values == 1,
        index=df_sorted.index
    )

    one_groups = pd.Series(
        s.values == 0,
        index=df_sorted.index
    ).cumsum()

    one_len = (
        is_one
        .groupby(one_groups)
        .transform('sum')
    )

    if 'abs_temp_diff_1' in df_sorted.columns:

        spike_in_group = (
            (df_sorted['abs_temp_diff_1'] >= 1.7)
            .groupby(one_groups)
            .transform('any')
        )

    else:

        spike_in_group = pd.Series(
            False,
            index=df_sorted.index
        )

    invalid_block = (
        is_one
        & (one_len >= 2)
        & (one_len <= 11)
        & ~spike_in_group
    )

    s = pd.Series(
        np.where(
            invalid_block,
            0,
            s.values
        ),
        index=df_sorted.index
    )

    return s.sort_index()


# =====================================================================
# 4. Threshold 탐색 함수
# =====================================================================

GLOBAL_BEST_TH = 0.19


def find_dynamic_thresholds(valid_df):

    best_thresholds = {}

    for station in valid_df['station'].unique():

        station_mask = (
            valid_df['station'] == station
        )

        layers = valid_df.loc[
            station_mask,
            'layer'
        ].unique()

        for layer in layers:

            mask = (
                (valid_df['station'] == station)
                &
                (valid_df['layer'] == layer)
            )

            y_true = valid_df.loc[
                mask,
                'label'
            ].values

            prob = valid_df.loc[
                mask,
                'prob'
            ].values

            # 해당 station/layer에 anomaly가 하나도 없는 경우
            if y_true.sum() == 0:

                best_thresholds[
                    (station, layer)
                ] = GLOBAL_BEST_TH

                continue

            best_f1 = -1
            best_th = GLOBAL_BEST_TH

            # 조금 더 촘촘하게 탐색
            for th in np.arange(
                0.01,
                0.81,
                0.01
            ):

                pred = (
                    prob >= th
                ).astype(int)

                f1 = f1_score(
                    y_true,
                    pred,
                    zero_division=0
                )

                if f1 > best_f1:

                    best_f1 = f1
                    best_th = th

            best_thresholds[
                (station, layer)
            ] = best_th

    return best_thresholds


# =====================================================================
# 5. Threshold 적용
# =====================================================================

def apply_dynamic_threshold(
    df,
    best_thresholds
):

    result = df.copy()

    result['base_pred'] = 0

    for station in result['station'].unique():

        for layer in result.loc[
            result['station'] == station,
            'layer'
        ].unique():

            mask = (
                (result['station'] == station)
                &
                (result['layer'] == layer)
            )

            prob = result.loc[
                mask,
                'prob'
            ].values

            th = best_thresholds.get(
                (station, layer),
                GLOBAL_BEST_TH
            )

            result.loc[
                mask,
                'base_pred'
            ] = (
                prob >= th
            ).astype(int)

    return result


# =====================================================================
# 6. 하나의 Validation 실험
# =====================================================================

def run_validation(
    scenario_name,
    scenario
):

    print("\n")
    print("=" * 80)
    print(f"VALIDATION : {scenario_name}")
    print("=" * 80)

    # -------------------------------------------------------------
    # Train
    # -------------------------------------------------------------

    train_mask = (
        (train_df['time'] >= scenario['train_start'])
        &
        (train_df['time'] <= scenario['train_end'])
    )

    model_train = train_df.loc[
        train_mask
    ].copy()

    # Validation 기간이 train에 포함되어 있다면 제거
    for start, end in scenario['exclude_from_train']:

        exclude_mask = (
            (model_train['time'] >= start)
            &
            (model_train['time'] <= end)
        )

        model_train = model_train.loc[
            ~exclude_mask
        ].copy()

    # -------------------------------------------------------------
    # Validation
    # -------------------------------------------------------------

    valid_mask = (
        (train_df['time'] >= scenario['valid_start'])
        &
        (train_df['time'] <= scenario['valid_end'])
    )

    valid = train_df.loc[
        valid_mask
    ].copy()

    print(
        f"Train : {model_train['time'].min()} ~ "
        f"{model_train['time'].max()}"
    )

    print(
        f"Valid : {valid['time'].min()} ~ "
        f"{valid['time'].max()}"
    )

    print(
        f"Train shape : {model_train.shape}"
    )

    print(
        f"Valid shape : {valid.shape}"
    )

    print(
        f"Train anomaly : "
        f"{model_train['label'].mean():.4%}"
    )

    print(
        f"Valid anomaly : "
        f"{valid['label'].mean():.4%}"
    )

    # -------------------------------------------------------------
    # X / y
    # -------------------------------------------------------------

    X_train = model_train[
        all_features
    ]

    y_train = model_train[
        'label'
    ]

    X_val = valid[
        all_features
    ]

    y_val = valid[
        'label'
    ].values

    # -------------------------------------------------------------
    # Class imbalance
    # -------------------------------------------------------------

    scale_pos_weight = (
        (y_train == 0).sum()
        /
        (y_train == 1).sum()
    )

    print(
        f"scale_pos_weight = "
        f"{scale_pos_weight:.3f}"
    )

    # -------------------------------------------------------------
    # XGBoost
    # -------------------------------------------------------------

    model = XGBClassifier(

        objective='binary:logistic',

        n_estimators=1000,

        learning_rate=0.03,

        max_depth=6,

        min_child_weight=5,

        subsample=0.8,

        colsample_bytree=0.8,

        scale_pos_weight=scale_pos_weight,

        random_state=42,

        n_jobs=-1,

        eval_metric='logloss'
    )

    print("XGBoost 학습 중...")

    model.fit(
        X_train,
        y_train
    )

    # -------------------------------------------------------------
    # Probability
    # -------------------------------------------------------------

    valid['prob'] = model.predict_proba(
        X_val
    )[:, 1]

    # -------------------------------------------------------------
    # 1. Global threshold 성능
    # -------------------------------------------------------------

    global_scores = []

    for th in np.arange(
        0.01,
        0.81,
        0.01
    ):

        pred = (
            valid['prob'].values >= th
        ).astype(int)

        f1 = f1_score(
            y_val,
            pred,
            zero_division=0
        )

        global_scores.append(
            (th, f1)
        )

    global_best_th, global_best_f1 = max(
        global_scores,
        key=lambda x: x[1]
    )

    # -------------------------------------------------------------
    # 2. Dynamic threshold
    # -------------------------------------------------------------

    best_thresholds = find_dynamic_thresholds(
        valid
    )

    valid = apply_dynamic_threshold(
        valid,
        best_thresholds
    )

    dynamic_f1_before_post = f1_score(
        valid['label'],
        valid['base_pred'],
        zero_division=0
    )

    # -------------------------------------------------------------
    # 3. Domain Rules
    # -------------------------------------------------------------

    valid['final_pred'] = 0

    for station in valid['station'].unique():

        for layer in valid.loc[
            valid['station'] == station,
            'layer'
        ].unique():

            mask = (
                (valid['station'] == station)
                &
                (valid['layer'] == layer)
            )

            valid.loc[
                mask,
                'final_pred'
            ] = apply_strict_domain_rules(
                valid.loc[mask]
            )

    dynamic_f1_after_post = f1_score(
        valid['label'],
        valid['final_pred'],
        zero_division=0
    )

    # -------------------------------------------------------------
    # Prediction statistics
    # -------------------------------------------------------------

    raw_ratio = (
        valid['prob'] >= global_best_th
    ).mean()

    dynamic_ratio = (
        valid['base_pred']
    ).mean()

    final_ratio = (
        valid['final_pred']
    ).mean()

    # -------------------------------------------------------------
    # 결과
    # -------------------------------------------------------------

    result = {

        'scenario': scenario_name,

        'train_n': len(model_train),

        'valid_n': len(valid),

        'train_anomaly': model_train['label'].mean(),

        'valid_anomaly': valid['label'].mean(),

        'global_best_th': global_best_th,

        'global_best_f1': global_best_f1,

        'dynamic_f1_before_post': dynamic_f1_before_post,

        'dynamic_f1_after_post': dynamic_f1_after_post,

        'global_pred_ratio': raw_ratio,

        'dynamic_pred_ratio': dynamic_ratio,

        'final_pred_ratio': final_ratio
    }

    print()
    print("-" * 80)
    print(f"Global threshold : {global_best_th:.2f}")
    print(f"Global F1        : {global_best_f1:.6f}")
    print(
        f"Dynamic F1       : "
        f"{dynamic_f1_before_post:.6f}"
    )
    print(
        f"Postprocess F1   : "
        f"{dynamic_f1_after_post:.6f}"
    )

    print()
    print(
        f"Global pred ratio  : "
        f"{raw_ratio:.4%}"
    )

    print(
        f"Dynamic pred ratio : "
        f"{dynamic_ratio:.4%}"
    )

    print(
        f"Final pred ratio   : "
        f"{final_ratio:.4%}"
    )

    print("-" * 80)

    return result, valid, model, best_thresholds


# =====================================================================
# 7. 모든 Validation 자동 실행
# =====================================================================

all_results = []

validation_outputs = {}

for scenario_name, scenario in validation_scenarios.items():

    result, valid_result, model, thresholds = run_validation(
        scenario_name,
        scenario
    )

    all_results.append(result)

    validation_outputs[
        scenario_name
    ] = {
        'valid': valid_result,
        'model': model,
        'thresholds': thresholds
    }


# =====================================================================
# 8. 결과 테이블
# =====================================================================

results_df = pd.DataFrame(
    all_results
)

results_df = results_df.sort_values(
    'dynamic_f1_after_post',
    ascending=False
)

print("\n")
print("=" * 100)
print("🔥 VALIDATION 결과 비교")
print("=" * 100)

print(
    results_df[
        [
            'scenario',
            'train_n',
            'valid_n',
            'valid_anomaly',
            'global_best_th',
            'global_best_f1',
            'dynamic_f1_before_post',
            'dynamic_f1_after_post',
            'final_pred_ratio'
        ]
    ].to_string(index=False)
)



데이터 로딩


ParserError: Error tokenizing data. C error: out of memory

In [ ]:
import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 2. Train / Validation 설정
#
# Train
#   - 2024 전체
#   - 2025-07 ~ 2025-12
#
# Validation
#   - 2025-01 ~ 2025-06
#
# Test 2026이 1~6월이므로 H1을 Validation으로 사용
# =====================================================================

model_train = train_df[
    (
        (train_df['time'] >= '2024-01-01')
        &
        (train_df['time'] <= '2024-12-31 23:59:59')
    )
    |
    (
        (train_df['time'] >= '2025-07-01')
        &
        (train_df['time'] <= '2025-12-31 23:59:59')
    )
].copy()


valid = train_df[
    (
        (train_df['time'] >= '2025-01-01')
        &
        (train_df['time'] <= '2025-06-30 23:59:59')
    )
].copy()


print("\nTrain shape:", model_train.shape)
print("Valid shape:", valid.shape)

print(
    "Train anomaly ratio:",
    f"{model_train['label'].mean():.4%}"
)

print(
    "Valid anomaly ratio:",
    f"{valid['label'].mean():.4%}"
)


print("\nTRAIN station:")
print(
    model_train['station']
    .value_counts()
)

print("\nVALID station:")
print(
    valid['station']
    .value_counts()
)


# =====================================================================
# 3. Feature 설정
# =====================================================================

drop_cols = [
    'label',
    'anomaly_type',
    'time',
    'dataset',
    'depth',
    'depth_diff',
    'station',
    'layer'
]


all_features = [
    c
    for c in train_df.columns
    if c not in drop_cols
]


print(
    f"\n사용 Feature 개수: "
    f"{len(all_features)}개"
)


X_train = model_train[
    all_features
]

y_train = model_train[
    'label'
]


X_val = valid[
    all_features
]

y_val = valid[
    'label'
].values


# =====================================================================
# 4. XGBoost
# =====================================================================

scale_pos_weight = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)


print(
    "\nscale_pos_weight:",
    scale_pos_weight
)


xgb_model = XGBClassifier(

    objective='binary:logistic',

    n_estimators=1000,

    learning_rate=0.03,

    max_depth=6,

    min_child_weight=5,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    n_jobs=-1,

    eval_metric='logloss'
)


print("\nXGBoost 학습 중...")


xgb_model.fit(
    X_train,
    y_train
)


# =====================================================================
# 5. Validation probability
# =====================================================================

valid['prob'] = xgb_model.predict_proba(
    X_val
)[:, 1]


print("\nValidation Probability")

print(
    valid['prob'].describe()
)


# =====================================================================
# 6. Global Threshold 확인
# =====================================================================

print("\n" + "=" * 80)
print("GLOBAL THRESHOLD")
print("=" * 80)


global_results = []


for th in np.arange(
    0.01,
    0.81,
    0.01
):

    pred = (
        valid['prob'].values >= th
    ).astype(int)

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    global_results.append(
        (th, f1)
    )


GLOBAL_BEST_TH, global_best_f1 = max(
    global_results,
    key=lambda x: x[1]
)


print(
    f"Global Best Threshold = "
    f"{GLOBAL_BEST_TH:.2f}"
)

print(
    f"Global Best F1 = "
    f"{global_best_f1:.6f}"
)


# =====================================================================
# 7. Station × Layer Dynamic Threshold
# =====================================================================

print("\n" + "=" * 80)
print("STATION × LAYER DYNAMIC THRESHOLD")
print("=" * 80)


best_thresholds = {}


for station in valid[
    'station'
].unique():

    station_layers = valid.loc[
        valid['station'] == station,
        'layer'
    ].unique()

    for layer in station_layers:

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )

        y_true_sl = valid.loc[
            mask,
            'label'
        ].values

        prob_sl = valid.loc[
            mask,
            'prob'
        ].values


        # anomaly 없는 layer
        if y_true_sl.sum() == 0:

            best_thresholds[
                (station, layer)
            ] = GLOBAL_BEST_TH

            continue


        best_f1 = -1
        best_th = GLOBAL_BEST_TH


        for th in np.arange(
            0.01,
            0.81,
            0.01
        ):

            pred = (
                prob_sl >= th
            ).astype(int)

            f1 = f1_score(
                y_true_sl,
                pred,
                zero_division=0
            )


            if f1 > best_f1:

                best_f1 = f1
                best_th = th


        best_thresholds[
            (station, layer)
        ] = best_th


        print(
            f"{station:8s} "
            f"Layer={layer} | "
            f"N={mask.sum():7d} | "
            f"Anomaly={y_true_sl.sum():5d} | "
            f"TH={best_th:.2f} | "
            f"F1={best_f1:.4f}"
        )


# =====================================================================
# 8. Dynamic Threshold 적용
# =====================================================================

valid['base_pred'] = 0


for station in valid[
    'station'
].unique():

    for layer in valid.loc[
        valid['station'] == station,
        'layer'
    ].unique():

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )


        th = best_thresholds.get(
            (station, layer),
            GLOBAL_BEST_TH
        )


        valid.loc[
            mask,
            'base_pred'
        ] = (
            valid.loc[
                mask,
                'prob'
            ].values
            >= th
        ).astype(int)


dynamic_f1 = f1_score(
    valid['label'],
    valid['base_pred'],
    zero_division=0
)


print(
    "\nDynamic Threshold F1:",
    f"{dynamic_f1:.6f}"
)
# =====================================================================
# 23. 후처리 Ablation Test
# =====================================================================

import itertools
import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 1. 후처리 함수
# =====================================================================

def apply_postprocess_ablation(
    df,
    use_flatline=False,
    use_gap=False,
    use_speckle=False
):

    df_sorted = df.sort_values(
        'time'
    ).copy()

    # 시작점은 기존 Dynamic Threshold 결과
    s = df_sorted[
        'base_pred'
    ].astype(int).copy()


    # =================================================================
    # Rule 1. Flatline
    # =================================================================

    if use_flatline:

        if 'flatline_length' in df_sorted.columns:

            s = pd.Series(
                np.where(
                    df_sorted[
                        'flatline_length'
                    ] >= 12,

                    1,

                    s.values
                ),
                index=df_sorted.index
            )


    # =================================================================
    # Rule 2. Gap Filling
    # =================================================================

    if use_gap:

        is_zero = pd.Series(
            s.values == 0,
            index=df_sorted.index
        )


        zero_groups = pd.Series(
            s.values != 0,
            index=df_sorted.index
        ).cumsum()


        zero_len = (
            is_zero
            .groupby(zero_groups)
            .transform('sum')
        )


        first_group = (
            zero_groups.iloc[0]
            if s.iloc[0] == 0
            else -1
        )


        last_group = (
            zero_groups.iloc[-1]
            if s.iloc[-1] == 0
            else -1
        )


        is_internal_zero = (
            is_zero
            &
            (zero_groups != first_group)
            &
            (zero_groups != last_group)
        )


        s = pd.Series(
            np.where(
                is_internal_zero
                &
                (zero_len <= 17),

                1,

                s.values
            ),
            index=df_sorted.index
        )


    # =================================================================
    # Rule 3. Speckle Removal
    # =================================================================

    if use_speckle:

        is_one = pd.Series(
            s.values == 1,
            index=df_sorted.index
        )


        one_groups = pd.Series(
            s.values == 0,
            index=df_sorted.index
        ).cumsum()


        one_len = (
            is_one
            .groupby(one_groups)
            .transform('sum')
        )


        # spike 보호
        if 'abs_temp_diff_1' in df_sorted.columns:

            spike_in_group = (
                (
                    df_sorted[
                        'abs_temp_diff_1'
                    ] >= 1.7
                )
                .groupby(
                    one_groups
                )
                .transform(
                    'any'
                )
            )

        else:

            spike_in_group = pd.Series(
                False,
                index=df_sorted.index
            )


        invalid_block = (
            is_one
            &
            (one_len >= 2)
            &
            (one_len <= 11)
            &
            ~spike_in_group
        )


        s = pd.Series(
            np.where(
                invalid_block,
                0,
                s.values
            ),
            index=df_sorted.index
        )


    return s.sort_index()


# =====================================================================
# 2. 실험 조합
# =====================================================================

postprocess_configs = [

    {
        'name': 'None',
        'flatline': False,
        'gap': False,
        'speckle': False
    },

    {
        'name': 'Flatline',
        'flatline': True,
        'gap': False,
        'speckle': False
    },

    {
        'name': 'Gap',
        'flatline': False,
        'gap': True,
        'speckle': False
    },

    {
        'name': 'Speckle',
        'flatline': False,
        'gap': False,
        'speckle': True
    },

    {
        'name': 'Flatline+Gap',
        'flatline': True,
        'gap': True,
        'speckle': False
    },

    {
        'name': 'Flatline+Speckle',
        'flatline': True,
        'gap': False,
        'speckle': True
    },

    {
        'name': 'Gap+Speckle',
        'flatline': False,
        'gap': True,
        'speckle': True
    },

    {
        'name': 'Flatline+Gap+Speckle',
        'flatline': True,
        'gap': True,
        'speckle': True
    }
]


# =====================================================================
# 3. 각 후처리 조합 실행
# =====================================================================

ablation_results = []


print("\n" + "=" * 120)
print("POSTPROCESS ABLATION TEST")
print("=" * 120)


for config in postprocess_configs:

    config_name = config[
        'name'
    ]

    print(
        f"\n실행 중: {config_name}"
    )


    pred_col = (
        'pred_'
        +
        config_name
        .replace('+', '_')
        .replace(' ', '_')
    )


    valid[
        pred_col
    ] = 0


    # -------------------------------------------------------------
    # station × layer 단위 적용
    # -------------------------------------------------------------

    for station in valid[
        'station'
    ].unique():

        layers = valid.loc[
            valid['station'] == station,
            'layer'
        ].unique()


        for layer in layers:

            mask = (
                (valid['station'] == station)
                &
                (valid['layer'] == layer)
            )


            valid.loc[
                mask,
                pred_col
            ] = apply_postprocess_ablation(

                valid.loc[
                    mask
                ],

                use_flatline=
                    config['flatline'],

                use_gap=
                    config['gap'],

                use_speckle=
                    config['speckle']
            )


    # -------------------------------------------------------------
    # 전체 성능
    # -------------------------------------------------------------

    y_true = valid[
        'label'
    ]

    y_pred = valid[
        pred_col
    ]


    overall_f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )


    overall_precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )


    overall_recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )


    overall_pred_ratio = (
        y_pred.mean()
    )


    # -------------------------------------------------------------
    # station별 성능
    # -------------------------------------------------------------

    station_f1_dict = {}


    for station in sorted(
        valid[
            'station'
        ].unique()
    ):

        mask = (
            valid['station']
            ==
            station
        )


        station_f1 = f1_score(
            valid.loc[
                mask,
                'label'
            ],
            valid.loc[
                mask,
                pred_col
            ],
            zero_division=0
        )


        station_f1_dict[
            station
        ] = station_f1


    # -------------------------------------------------------------
    # 결과 저장
    # -------------------------------------------------------------

    result = {

        'config':
            config_name,

        'overall_f1':
            overall_f1,

        'precision':
            overall_precision,

        'recall':
            overall_recall,

        'pred_ratio':
            overall_pred_ratio
    }


    # station 자동 추가
    for station, f1 in station_f1_dict.items():

        result[
            f'{station}_f1'
        ] = f1


    ablation_results.append(
        result
    )


# =====================================================================
# 4. 결과 DataFrame
# =====================================================================

ablation_df = pd.DataFrame(
    ablation_results
)


ablation_df = ablation_df.sort_values(
    'overall_f1',
    ascending=False
).reset_index(
    drop=True
)


# =====================================================================
# 5. 전체 결과 출력
# =====================================================================

print("\n" + "=" * 140)
print("POSTPROCESS ABLATION 결과")
print("=" * 140)


print(
    ablation_df.to_string(
        index=False
    )
)


# =====================================================================
# 6. 최고 조합 출력
# =====================================================================

best_row = (
    ablation_df.iloc[0]
)


print("\n" + "=" * 100)
print("최고 후처리 조합")
print("=" * 100)


print(
    f"Best Config   : "
    f"{best_row['config']}"
)

print(
    f"Overall F1    : "
    f"{best_row['overall_f1']:.6f}"
)

print(
    f"Precision     : "
    f"{best_row['precision']:.6f}"
)

print(
    f"Recall        : "
    f"{best_row['recall']:.6f}"
)

print(
    f"Pred Ratio    : "
    f"{best_row['pred_ratio']:.4%}"
)


# =====================================================================
# 7. Station × Layer별로 조합 비교
# =====================================================================

print("\n" + "=" * 160)
print("STATION × LAYER별 후처리 조합 비교")
print("=" * 160)


layer_ablation_rows = []


for station in sorted(
    valid[
        'station'
    ].unique()
):

    layers = sorted(
        valid.loc[
            valid['station'] == station,
            'layer'
        ].unique()
    )


    for layer in layers:

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )


        y_true = valid.loc[
            mask,
            'label'
        ]


        for config in postprocess_configs:

            config_name = config[
                'name'
            ]

            pred_col = (
                'pred_'
                +
                config_name
                .replace('+', '_')
                .replace(' ', '_')
            )


            y_pred = valid.loc[
                mask,
                pred_col
            ]


            layer_f1 = f1_score(
                y_true,
                y_pred,
                zero_division=0
            )


            layer_precision = (
                precision_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            )


            layer_recall = (
                recall_score(
                    y_true,
                    y_pred,
                    zero_division=0
                )
            )


            layer_ablation_rows.append({

                'station':
                    station,

                'layer':
                    layer,

                'config':
                    config_name,

                'N':
                    mask.sum(),

                'true_anomaly':
                    int(
                        y_true.sum()
                    ),

                'pred_ratio':
                    y_pred.mean(),

                'precision':
                    layer_precision,

                'recall':
                    layer_recall,

                'f1':
                    layer_f1
            })


layer_ablation_df = pd.DataFrame(
    layer_ablation_rows
)


# =====================================================================
# 8. 각 Station × Layer의 최고 후처리 조합
# =====================================================================

best_layer_config = (

    layer_ablation_df

    .sort_values(
        'f1',
        ascending=False
    )

    .groupby(
        [
            'station',
            'layer'
        ],
        as_index=False
    )

    .first()

)


print(
    best_layer_config[
        [
            'station',
            'layer',
            'config',
            'N',
            'true_anomaly',
            'pred_ratio',
            'precision',
            'recall',
            'f1'
        ]
    ]
    .sort_values(
        [
            'station',
            'layer'
        ]
    )
    .to_string(
        index=False
    )
)





Train shape: (568613, 158)
Valid shape: (208093, 158)
Train anomaly ratio: 3.9069%
Valid anomaly ratio: 4.7628%

TRAIN station:
station
S-ORS    432196
I-ORS    126844
G-ORS      9573
Name: count, dtype: int64

VALID station:
station
S-ORS    126642
I-ORS     64521
G-ORS     16930
Name: count, dtype: int64

사용 Feature 개수: 150개

scale_pos_weight: 24.595903668692326

XGBoost 학습 중...

Validation Probability
count    2.080930e+05
mean     3.263662e-02
std      1.488989e-01
min      1.624634e-08
25%      2.451296e-04
50%      9.163268e-04
75%      3.900779e-03
max      9.999787e-01
Name: prob, dtype: float64

GLOBAL THRESHOLD
Global Best Threshold = 0.32
Global Best F1 = 0.581212

STATION × LAYER DYNAMIC THRESHOLD
G-ORS    Layer=1 | N=  16930 | Anomaly=   72 | TH=0.17 | F1=0.2462
I-ORS    Layer=1 | N=  19227 | Anomaly=  973 | TH=0.74 | F1=0.3955
I-ORS    Layer=2 | N=   5802 | Anomaly=  301 | TH=0.04 | F1=0.4488
I-ORS    Layer=3 | N=   5804 | Anomaly=  257 | TH=0.80 | F1=0.6481
I-ORS    Lay

In [ ]:
import joblib

joblib.dump(
    xgb_model,
    "final_xgb_model.pkl"
)

In [16]:
# =====================================================================
# 24. Anomaly Type별 성능 분석
# =====================================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)
valid['final_pred'] = valid['pred_Speckle'].copy()

print("\n" + "=" * 120)
print("ANOMALY TYPE별 VALIDATION 성능")
print("=" * 120)

type_rows = []

# anomaly_type NaN은 normal일 가능성이 있으므로
# 여기서는 anomaly_type이 실제로 존재하는 이상치만 분석
anomaly_valid = valid[
    valid['anomaly_type'].notna()
].copy()

for anomaly_type in sorted(
    anomaly_valid['anomaly_type'].unique()
):

    mask = (
        anomaly_valid['anomaly_type']
        ==
        anomaly_type
    )

    y_true = anomaly_valid.loc[
        mask,
        'label'
    ]

    y_pred = anomaly_valid.loc[
        mask,
        'final_pred'
    ]

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    type_rows.append({
        'anomaly_type': anomaly_type,
        'N': mask.sum(),
        'true_anomaly': int(y_true.sum()),
        'pred_anomaly': int(y_pred.sum()),
        'precision': precision,
        'recall': recall,
        'f1': f1
    })


anomaly_type_result = pd.DataFrame(
    type_rows
)

print(
    anomaly_type_result
    .sort_values(
        'recall'
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 25. Station × Anomaly Type별 Recall
# =====================================================================

print("\n" + "=" * 140)
print("STATION × ANOMALY TYPE별 성능")
print("=" * 140)

station_type_rows = []

for station in sorted(
    anomaly_valid['station'].unique()
):

    station_df = anomaly_valid[
        anomaly_valid['station'] == station
    ]

    for anomaly_type in sorted(
        station_df['anomaly_type'].unique()
    ):

        mask = (
            (anomaly_valid['station'] == station)
            &
            (anomaly_valid['anomaly_type'] == anomaly_type)
        )

        y_true = anomaly_valid.loc[
            mask,
            'label'
        ]

        y_pred = anomaly_valid.loc[
            mask,
            'final_pred'
        ]

        precision = precision_score(
            y_true,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_true,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0
        )

        station_type_rows.append({
            'station': station,
            'anomaly_type': anomaly_type,
            'N': mask.sum(),
            'true_anomaly': int(y_true.sum()),
            'pred_anomaly': int(y_pred.sum()),
            'precision': precision,
            'recall': recall,
            'f1': f1
        })


station_type_result = pd.DataFrame(
    station_type_rows
)

print(
    station_type_result
    .sort_values(
        ['station', 'recall']
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 26. Station × Layer × Anomaly Type별 Recall
# =====================================================================

print("\n" + "=" * 170)
print("STATION × LAYER × ANOMALY TYPE별 성능")
print("=" * 170)

layer_type_rows = []

for station in sorted(
    anomaly_valid['station'].unique()
):

    layers = sorted(
        anomaly_valid.loc[
            anomaly_valid['station'] == station,
            'layer'
        ].unique()
    )

    for layer in layers:

        layer_df = anomaly_valid[
            (anomaly_valid['station'] == station)
            &
            (anomaly_valid['layer'] == layer)
        ]

        for anomaly_type in sorted(
            layer_df['anomaly_type'].unique()
        ):

            mask = (
                (anomaly_valid['station'] == station)
                &
                (anomaly_valid['layer'] == layer)
                &
                (anomaly_valid['anomaly_type'] == anomaly_type)
            )

            y_true = anomaly_valid.loc[
                mask,
                'label'
            ]

            y_pred = anomaly_valid.loc[
                mask,
                'final_pred'
            ]

            precision = precision_score(
                y_true,
                y_pred,
                zero_division=0
            )

            recall = recall_score(
                y_true,
                y_pred,
                zero_division=0
            )

            f1 = f1_score(
                y_true,
                y_pred,
                zero_division=0
            )

            layer_type_rows.append({
                'station': station,
                'layer': layer,
                'anomaly_type': anomaly_type,
                'N': mask.sum(),
                'true_anomaly': int(y_true.sum()),
                'pred_anomaly': int(y_pred.sum()),
                'precision': precision,
                'recall': recall,
                'f1': f1
            })


layer_type_result = pd.DataFrame(
    layer_type_rows
)


# Recall 낮은 순으로 먼저 보기
print(
    layer_type_result
    .sort_values(
        ['recall', 'N'],
        ascending=[True, False]
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 27. Recall 낮은 케이스만 따로 보기
# =====================================================================

print("\n" + "=" * 140)
print("RECALL < 0.5 인 Station × Layer × Anomaly Type")
print("=" * 140)

low_recall = layer_type_result[
    layer_type_result['recall'] < 0.5
].copy()

print(
    low_recall
    .sort_values(
        ['recall', 'true_anomaly'],
        ascending=[True, False]
    )
    .to_string(
        index=False
    )
)




ANOMALY TYPE별 VALIDATION 성능
   anomaly_type    N  true_anomaly  pred_anomaly  precision   recall       f1
   drift+offset   45            45             0        0.0 0.000000 0.000000
          drift 3354          3354          1127        1.0 0.336017 0.503013
 flatline+drift  150           150            55        1.0 0.366667 0.536585
         offset 2711          2711          1242        1.0 0.458134 0.628384
          noise 1902          1902          1776        1.0 0.933754 0.965742
          spike   21            21            20        1.0 0.952381 0.975610
       flatline 1691          1691          1669        1.0 0.986990 0.993452
offset+flatline   36            36            36        1.0 1.000000 1.000000
    spike+drift    1             1             1        1.0 1.000000 1.000000

STATION × ANOMALY TYPE별 성능
station    anomaly_type    N  true_anomaly  pred_anomaly  precision   recall       f1
  G-ORS           noise   68            68            50        1.0 0.735294 

In [17]:
# =====================================================================
# LONG ANOMALY의 probability 분포 확인
# =====================================================================

LONG_TYPES = [
    'drift',
    'offset',
    'drift+offset',
    'flatline+drift'
]

long_df = valid[
    valid['anomaly_type'].isin(LONG_TYPES)
].copy()

print("\n" + "=" * 140)
print("LONG ANOMALY PROBABILITY 분포")
print("=" * 140)

rows = []

for station in sorted(long_df['station'].unique()):

    layers = sorted(
        long_df.loc[
            long_df['station'] == station,
            'layer'
        ].unique()
    )

    for layer in layers:

        types = long_df.loc[
            (long_df['station'] == station)
            &
            (long_df['layer'] == layer),
            'anomaly_type'
        ].unique()

        for anomaly_type in sorted(types):

            mask = (
                (long_df['station'] == station)
                &
                (long_df['layer'] == layer)
                &
                (long_df['anomaly_type'] == anomaly_type)
            )

            p = long_df.loc[
                mask,
                'prob'
            ]

            rows.append({
                'station': station,
                'layer': layer,
                'anomaly_type': anomaly_type,
                'N': len(p),

                'prob_mean': p.mean(),
                'prob_median': p.median(),

                'prob_q25': p.quantile(0.25),
                'prob_q75': p.quantile(0.75),

                'prob_q90': p.quantile(0.90),

                'prob_max': p.max(),

                'current_threshold':
                    best_thresholds.get(
                        (station, layer),
                        GLOBAL_BEST_TH
                    )
            })


long_prob_result = pd.DataFrame(rows)

print(
    long_prob_result
    .sort_values(
        ['prob_median', 'N'],
        ascending=[True, False]
    )
    .to_string(index=False)
)


LONG ANOMALY PROBABILITY 분포
station  layer   anomaly_type   N  prob_mean  prob_median  prob_q25  prob_q75  prob_q90  prob_max  current_threshold
  I-ORS      7         offset 575   0.035879     0.000431  0.000205  0.024742  0.078427  0.406889               0.02
  I-ORS      1   drift+offset  45   0.000953     0.000896  0.000731  0.001097  0.001207  0.003553               0.74
  S-ORS      8          drift 614   0.007672     0.001750  0.000339  0.006062  0.014288  0.862004               0.01
  I-ORS      1          drift 339   0.078363     0.002377  0.000772  0.015533  0.054401  0.956947               0.74
  S-ORS      8         offset 489   0.009345     0.002846  0.001063  0.009803  0.025875  0.351737               0.01
  I-ORS      1         offset 346   0.015559     0.003372  0.001528  0.009604  0.032884  0.748164               0.74
  I-ORS      7          drift 782   0.072268     0.008637  0.001102  0.040880  0.406809  0.683033               0.02
  S-ORS      3         offset 121  

In [ ]:
# =====================================================================
# 29. LONG ANOMALY 전용 XGBoost
#     - 기존 150 Features 그대로 사용
#     - Drift / Offset 계열만 positive
# =====================================================================

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 0. 현재 상태 확인
# =====================================================================

assert 'final_pred' in valid.columns, \
    "valid['final_pred']가 없습니다. Speckle-only final_pred를 먼저 생성하세요."

assert 'anomaly_type' in model_train.columns
assert 'anomaly_type' in valid.columns

print("=" * 100)
print("LONG ANOMALY MODEL TEST")
print("=" * 100)

print(
    f"현재 Base(Speckle) F1 = "
    f"{f1_score(valid['label'], valid['final_pred'], zero_division=0):.6f}"
)


# =====================================================================
# 1. LONG target 정의
#
# drift 또는 offset이 들어간 유형은 전부 LONG으로 취급
#
# drift
# offset
# drift+offset
# flatline+drift
# offset+flatline
# spike+drift
# =====================================================================

LONG_TYPES = [
    'drift',
    'offset',
    'drift+offset',
    'flatline+drift',
    'offset+flatline',
    'spike+drift'
]


model_train['long_label'] = (
    model_train['anomaly_type']
    .isin(LONG_TYPES)
    .astype(int)
)


valid['long_label'] = (
    valid['anomaly_type']
    .isin(LONG_TYPES)
    .astype(int)
)


print("\nLONG label 분포")

print("\nTRAIN")
print(
    model_train[
        'long_label'
    ].value_counts()
)

print(
    f"Train LONG ratio = "
    f"{model_train['long_label'].mean():.4%}"
)


print("\nVALID")
print(
    valid[
        'long_label'
    ].value_counts()
)

print(
    f"Valid LONG ratio = "
    f"{valid['long_label'].mean():.4%}"
)


# =====================================================================
# 2. Long Model 학습 데이터
# =====================================================================

X_train_long = model_train[
    all_features
]

y_train_long = model_train[
    'long_label'
]


X_val_long = valid[
    all_features
]

y_val_long = valid[
    'long_label'
]


# =====================================================================
# 3. Long Class Weight
# =====================================================================

long_scale_pos_weight = (
    (y_train_long == 0).sum()
    /
    (y_train_long == 1).sum()
)


print(
    "\nLong scale_pos_weight =",
    long_scale_pos_weight
)


# =====================================================================
# 4. Long XGBoost
#
# 일단 기존 XGB와 동일한 구조
# → 타겟 분리 자체의 효과만 확인
# =====================================================================

long_model = XGBClassifier(

    objective='binary:logistic',

    n_estimators=1000,

    learning_rate=0.03,

    max_depth=6,

    min_child_weight=5,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=long_scale_pos_weight,

    random_state=42,

    n_jobs=-1,

    eval_metric='logloss'
)


print("\nLONG XGBoost 학습 중...")


long_model.fit(
    X_train_long,
    y_train_long
)


# =====================================================================
# 5. Long probability
# =====================================================================

valid['long_prob'] = (
    long_model
    .predict_proba(
        X_val_long
    )[:, 1]
)


print("\nLONG Probability")

print(
    valid[
        'long_prob'
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# =====================================================================
# 6. Base prediction 저장
#
# final_pred = 앞에서 선택한 Speckle-only 결과
# =====================================================================

valid[
    'base_final_pred'
] = valid[
    'final_pred'
].astype(int)


BASE_F1 = f1_score(
    valid['label'],
    valid['base_final_pred'],
    zero_division=0
)


BASE_PRECISION = precision_score(
    valid['label'],
    valid['base_final_pred'],
    zero_division=0
)


BASE_RECALL = recall_score(
    valid['label'],
    valid['base_final_pred'],
    zero_division=0
)


print("\n" + "=" * 100)
print("BASE MODEL")
print("=" * 100)

print(
    f"F1        = {BASE_F1:.6f}"
)

print(
    f"Precision = {BASE_PRECISION:.6f}"
)

print(
    f"Recall    = {BASE_RECALL:.6f}"
)


# =====================================================================
# 7. Long Threshold 자동 탐색
#
# 핵심:
# Long model 자체 F1 최대화가 아니라
#
# Base OR Long
#
# 했을 때 "전체 label F1"이 최대가 되는 threshold 탐색
# =====================================================================

threshold_results = []


for th in np.arange(
    0.01,
    0.81,
    0.01
):

    long_pred = (
        valid['long_prob']
        >= th
    ).astype(int)


    # -------------------------------------------------------------
    # Base OR Long
    # -------------------------------------------------------------

    combined_pred = (
        (
            valid['base_final_pred'] == 1
        )
        |
        (
            long_pred == 1
        )
    ).astype(int)


    overall_f1 = f1_score(
        valid['label'],
        combined_pred,
        zero_division=0
    )


    precision = precision_score(
        valid['label'],
        combined_pred,
        zero_division=0
    )


    recall = recall_score(
        valid['label'],
        combined_pred,
        zero_division=0
    )


    # -------------------------------------------------------------
    # LONG anomaly recall
    # -------------------------------------------------------------

    long_mask = (
        valid['long_label'] == 1
    )


    long_recall = recall_score(
        valid.loc[
            long_mask,
            'label'
        ],
        combined_pred[
            long_mask
        ],
        zero_division=0
    )


    # -------------------------------------------------------------
    # drift recall
    # -------------------------------------------------------------

    drift_mask = (
        valid['anomaly_type']
        ==
        'drift'
    )


    if drift_mask.sum() > 0:

        drift_recall = (
            combined_pred[
                drift_mask
            ].mean()
        )

    else:

        drift_recall = np.nan


    # -------------------------------------------------------------
    # offset recall
    # -------------------------------------------------------------

    offset_mask = (
        valid['anomaly_type']
        ==
        'offset'
    )


    if offset_mask.sum() > 0:

        offset_recall = (
            combined_pred[
                offset_mask
            ].mean()
        )

    else:

        offset_recall = np.nan


    threshold_results.append({

        'threshold':
            th,

        'overall_f1':
            overall_f1,

        'precision':
            precision,

        'recall':
            recall,

        'long_recall':
            long_recall,

        'drift_recall':
            drift_recall,

        'offset_recall':
            offset_recall,

        'pred_ratio':
            combined_pred.mean(),

        'long_only_ratio':
            long_pred.mean()
    })


long_threshold_df = pd.DataFrame(
    threshold_results
)


# =====================================================================
# 8. Overall F1 기준 Best Threshold
# =====================================================================

best_idx = (
    long_threshold_df[
        'overall_f1'
    ].idxmax()
)


best_long_row = (
    long_threshold_df
    .loc[
        best_idx
    ]
)


BEST_LONG_TH = (
    best_long_row[
        'threshold'
    ]
)


print("\n" + "=" * 120)
print("LONG THRESHOLD 탐색 결과")
print("=" * 120)


print(
    f"Base F1              : "
    f"{BASE_F1:.6f}"
)

print(
    f"Best Long Threshold  : "
    f"{BEST_LONG_TH:.2f}"
)

print(
    f"Combined F1          : "
    f"{best_long_row['overall_f1']:.6f}"
)

print(
    f"Precision            : "
    f"{best_long_row['precision']:.6f}"
)

print(
    f"Recall               : "
    f"{best_long_row['recall']:.6f}"
)

print(
    f"Long Recall          : "
    f"{best_long_row['long_recall']:.6f}"
)

print(
    f"Drift Recall         : "
    f"{best_long_row['drift_recall']:.6f}"
)

print(
    f"Offset Recall        : "
    f"{best_long_row['offset_recall']:.6f}"
)

print(
    f"Prediction Ratio     : "
    f"{best_long_row['pred_ratio']:.4%}"
)


print(
    f"\nF1 변화 = "
    f"{best_long_row['overall_f1'] - BASE_F1:+.6f}"
)


# =====================================================================
# 9. 상위 Threshold 15개 확인
# =====================================================================

print("\n" + "=" * 140)
print("Overall F1 상위 Threshold")
print("=" * 140)


print(
    long_threshold_df
    .sort_values(
        'overall_f1',
        ascending=False
    )
    .head(15)
    .to_string(
        index=False
    )
)


# =====================================================================
# 10. Best Threshold로 최종 Combined Prediction
# =====================================================================

valid[
    'long_pred'
] = (
    valid[
        'long_prob'
    ]
    >= BEST_LONG_TH
).astype(int)


valid[
    'combined_pred'
] = (
    (
        valid[
            'base_final_pred'
        ] == 1
    )
    |
    (
        valid[
            'long_pred'
        ] == 1
    )
).astype(int)


# =====================================================================
# 11. Base vs Combined Station별 비교
# =====================================================================

print("\n" + "=" * 130)
print("BASE VS BASE+LONG : STATION별")
print("=" * 130)


station_compare_rows = []


for station in sorted(
    valid['station'].unique()
):

    mask = (
        valid['station']
        ==
        station
    )


    y_true = valid.loc[
        mask,
        'label'
    ]


    base_pred = valid.loc[
        mask,
        'base_final_pred'
    ]


    combined_pred = valid.loc[
        mask,
        'combined_pred'
    ]


    base_f1 = f1_score(
        y_true,
        base_pred,
        zero_division=0
    )


    new_f1 = f1_score(
        y_true,
        combined_pred,
        zero_division=0
    )


    station_compare_rows.append({

        'station':
            station,

        'N':
            mask.sum(),

        'actual_ratio':
            y_true.mean(),

        'base_pred_ratio':
            base_pred.mean(),

        'combined_pred_ratio':
            combined_pred.mean(),

        'base_f1':
            base_f1,

        'combined_f1':
            new_f1,

        'delta_f1':
            new_f1 - base_f1,

        'combined_precision':
            precision_score(
                y_true,
                combined_pred,
                zero_division=0
            ),

        'combined_recall':
            recall_score(
                y_true,
                combined_pred,
                zero_division=0
            )
    })


station_long_compare = pd.DataFrame(
    station_compare_rows
)


print(
    station_long_compare
    .to_string(
        index=False
    )
)


# =====================================================================
# 12. Anomaly Type Recall : Base VS Combined
# =====================================================================

print("\n" + "=" * 140)
print("ANOMALY TYPE RECALL : BASE VS BASE+LONG")
print("=" * 140)


type_compare_rows = []


# label=1인 이상 데이터만
anomaly_df = valid[
    valid['label'] == 1
].copy()


for anomaly_type in sorted(
    anomaly_df[
        'anomaly_type'
    ].dropna().unique()
):

    mask = (
        anomaly_df[
            'anomaly_type'
        ]
        ==
        anomaly_type
    )


    base_recall = (
        anomaly_df.loc[
            mask,
            'base_final_pred'
        ].mean()
    )


    combined_recall = (
        anomaly_df.loc[
            mask,
            'combined_pred'
        ].mean()
    )


    type_compare_rows.append({

        'anomaly_type':
            anomaly_type,

        'N':
            mask.sum(),

        'base_recall':
            base_recall,

        'combined_recall':
            combined_recall,

        'delta_recall':
            combined_recall
            -
            base_recall
    })


type_long_compare = pd.DataFrame(
    type_compare_rows
)


print(
    type_long_compare
    .sort_values(
        'base_recall'
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 13. 가장 문제였던 Station × Layer × Long Type 비교
# =====================================================================

print("\n" + "=" * 170)
print("STATION × LAYER × LONG TYPE : BASE VS BASE+LONG")
print("=" * 170)


long_detail_rows = []


long_valid = valid[
    valid[
        'anomaly_type'
    ].isin(
        LONG_TYPES
    )
].copy()


for station in sorted(
    long_valid['station'].unique()
):

    layers = sorted(
        long_valid.loc[
            long_valid['station'] == station,
            'layer'
        ].unique()
    )


    for layer in layers:

        types = sorted(
            long_valid.loc[
                (
                    long_valid['station'] == station
                )
                &
                (
                    long_valid['layer'] == layer
                ),
                'anomaly_type'
            ].unique()
        )


        for anomaly_type in types:

            mask = (
                (long_valid['station'] == station)
                &
                (long_valid['layer'] == layer)
                &
                (
                    long_valid['anomaly_type']
                    ==
                    anomaly_type
                )
            )


            base_recall = (
                long_valid.loc[
                    mask,
                    'base_final_pred'
                ].mean()
            )


            combined_recall = (
                long_valid.loc[
                    mask,
                    'combined_pred'
                ].mean()
            )


            long_prob_median = (
                long_valid.loc[
                    mask,
                    'long_prob'
                ].median()
            )


            long_detail_rows.append({

                'station':
                    station,

                'layer':
                    layer,

                'anomaly_type':
                    anomaly_type,

                'N':
                    mask.sum(),

                'base_recall':
                    base_recall,

                'combined_recall':
                    combined_recall,

                'delta_recall':
                    combined_recall
                    -
                    base_recall,

                'long_prob_median':
                    long_prob_median
            })


long_detail_compare = pd.DataFrame(
    long_detail_rows
)


print(
    long_detail_compare
    .sort_values(
        [
            'base_recall',
            'N'
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(
        index=False
    )
)





LONG ANOMALY MODEL TEST
현재 Base(Speckle) F1 = 0.634509

LONG label 분포

TRAIN
long_label
0    558807
1      9806
Name: count, dtype: int64
Train LONG ratio = 1.7245%

VALID
long_label
0    201796
1      6297
Name: count, dtype: int64
Valid LONG ratio = 3.0261%

Long scale_pos_weight = 56.986232918621255

LONG XGBoost 학습 중...

LONG Probability
count    2.080930e+05
mean     1.603316e-02
std      1.020826e-01
min      1.190555e-11
50%      3.197660e-05
75%      3.713959e-04
90%      2.941106e-03
95%      1.499059e-02
99%      7.121417e-01
max      9.999260e-01
Name: long_prob, dtype: float64

BASE MODEL
F1        = 0.634509
Precision = 0.675867
Recall    = 0.597922

LONG THRESHOLD 탐색 결과
Base F1              : 0.634509
Best Long Threshold  : 0.80
Combined F1          : 0.629234
Precision            : 0.661658
Recall               : 0.599839
Long Recall          : 0.393838
Drift Recall         : 0.337806
Offset Recall        : 0.458134
Prediction Ratio     : 4.3178%

F1 변화 = -0.005276

Over

In [19]:
# =====================================================================
# LONG MODEL v2
# station + layer 정보 추가
# =====================================================================

from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

# =====================================================================
# 1. Long label
# =====================================================================

LONG_TYPES = [
    'drift',
    'offset',
    'drift+offset',
    'flatline+drift',
    'offset+flatline',
    'spike+drift'
]

model_train['long_label'] = (
    model_train['anomaly_type']
    .isin(LONG_TYPES)
    .astype(int)
)

valid['long_label'] = (
    valid['anomaly_type']
    .isin(LONG_TYPES)
    .astype(int)
)


# =====================================================================
# 2. 기존 150 features
# =====================================================================

X_train_long = model_train[
    all_features
].copy()

X_val_long = valid[
    all_features
].copy()


# =====================================================================
# 3. Station / Layer One-Hot
# =====================================================================

train_cat = pd.get_dummies(
    model_train[
        ['station', 'layer']
    ].astype(str),
    prefix=[
        'station',
        'layer'
    ]
)

val_cat = pd.get_dummies(
    valid[
        ['station', 'layer']
    ].astype(str),
    prefix=[
        'station',
        'layer'
    ]
)


# Train 기준으로 컬럼 맞추기
train_cat, val_cat = train_cat.align(
    val_cat,
    join='left',
    axis=1,
    fill_value=0
)


X_train_long = pd.concat(
    [
        X_train_long.reset_index(drop=True),
        train_cat.reset_index(drop=True)
    ],
    axis=1
)

X_val_long = pd.concat(
    [
        X_val_long.reset_index(drop=True),
        val_cat.reset_index(drop=True)
    ],
    axis=1
)


y_train_long = (
    model_train['long_label']
    .reset_index(drop=True)
)


print(
    "Long feature 개수:",
    X_train_long.shape[1]
)

print(
    "\n추가된 categorical feature:"
)

print(
    train_cat.columns.tolist()
)


# =====================================================================
# 4. Weight
# =====================================================================

long_scale_pos_weight = (
    (y_train_long == 0).sum()
    /
    (y_train_long == 1).sum()
)

print(
    "\nscale_pos_weight:",
    long_scale_pos_weight
)


# =====================================================================
# 5. Long XGB v2
# =====================================================================

long_model_v2 = XGBClassifier(

    objective='binary:logistic',

    n_estimators=1000,

    learning_rate=0.03,

    max_depth=6,

    min_child_weight=5,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=long_scale_pos_weight,

    random_state=42,

    n_jobs=-1,

    eval_metric='logloss'
)


print("\nLong XGB v2 학습 중...")


long_model_v2.fit(
    X_train_long,
    y_train_long
)


# =====================================================================
# 6. Validation probability
# =====================================================================

valid[
    'long_prob_v2'
] = long_model_v2.predict_proba(
    X_val_long
)[:, 1]


# =====================================================================
# 7. Base prediction
# =====================================================================

base_pred = (
    valid['final_pred']
    .astype(int)
    .values
)

y_true = (
    valid['label']
    .astype(int)
    .values
)


BASE_F1 = f1_score(
    y_true,
    base_pred,
    zero_division=0
)


# =====================================================================
# 8. Threshold 탐색
# =====================================================================

rows = []


for th in np.arange(
    0.01,
    0.96,
    0.01
):

    long_pred = (
        valid[
            'long_prob_v2'
        ].values
        >= th
    ).astype(int)


    combined = (
        (base_pred == 1)
        |
        (long_pred == 1)
    ).astype(int)


    rows.append({

        'threshold':
            th,

        'f1':
            f1_score(
                y_true,
                combined,
                zero_division=0
            ),

        'precision':
            precision_score(
                y_true,
                combined,
                zero_division=0
            ),

        'recall':
            recall_score(
                y_true,
                combined,
                zero_division=0
            ),

        'pred_ratio':
            combined.mean()
    })


long_v2_threshold = pd.DataFrame(
    rows
)


best = long_v2_threshold.loc[
    long_v2_threshold[
        'f1'
    ].idxmax()
]


BEST_LONG_V2_TH = best[
    'threshold'
]


print("\n" + "=" * 100)

print(
    f"BASE F1       = "
    f"{BASE_F1:.6f}"
)

print(
    f"BEST LONG TH  = "
    f"{BEST_LONG_V2_TH:.2f}"
)

print(
    f"COMBINED F1   = "
    f"{best['f1']:.6f}"
)

print(
    f"DELTA F1      = "
    f"{best['f1'] - BASE_F1:+.6f}"
)

print(
    f"Precision     = "
    f"{best['precision']:.6f}"
)

print(
    f"Recall        = "
    f"{best['recall']:.6f}"
)

print("=" * 100)


# =====================================================================
# 9. 최종 combined prediction
# =====================================================================

valid[
    'long_pred_v2'
] = (
    valid[
        'long_prob_v2'
    ]
    >= BEST_LONG_V2_TH
).astype(int)


valid[
    'combined_pred_v2'
] = (
    (
        valid[
            'final_pred'
        ] == 1
    )
    |
    (
        valid[
            'long_pred_v2'
        ] == 1
    )
).astype(int)


# =====================================================================
# 10. 문제 Long Type들의 변화 확인
# =====================================================================

print("\n" + "=" * 145)
print("문제 LONG 유형 : BASE VS LONG-V2")
print("=" * 145)


problem_rows = []


for station in sorted(
    valid['station'].unique()
):

    for layer in sorted(
        valid.loc[
            valid['station'] == station,
            'layer'
        ].unique()
    ):

        temp = valid[
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
            &
            (valid['anomaly_type'].isin(
                LONG_TYPES
            ))
        ]


        if len(temp) == 0:

            continue


        for anomaly_type in sorted(
            temp[
                'anomaly_type'
            ].unique()
        ):

            x = temp[
                temp[
                    'anomaly_type'
                ]
                ==
                anomaly_type
            ]


            problem_rows.append({

                'station':
                    station,

                'layer':
                    layer,

                'type':
                    anomaly_type,

                'N':
                    len(x),

                'base_recall':
                    x[
                        'final_pred'
                    ].mean(),

                'v2_recall':
                    x[
                        'combined_pred_v2'
                    ].mean(),

                'long_prob_median':
                    x[
                        'long_prob_v2'
                    ].median()
            })


problem_result = pd.DataFrame(
    problem_rows
)


problem_result[
    'delta_recall'
] = (
    problem_result[
        'v2_recall'
    ]
    -
    problem_result[
        'base_recall'
    ]
)


print(
    problem_result
    .sort_values(
        [
            'base_recall',
            'N'
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(
        index=False
    )
)

Long feature 개수: 161

추가된 categorical feature:
['station_G-ORS', 'station_I-ORS', 'station_S-ORS', 'layer_1', 'layer_2', 'layer_3', 'layer_4', 'layer_5', 'layer_6', 'layer_7', 'layer_8']

scale_pos_weight: 56.986232918621255

Long XGB v2 학습 중...

BASE F1       = 0.634509
BEST LONG TH  = 0.95
COMBINED F1   = 0.633627
DELTA F1      = -0.000882
Precision     = 0.673869
Recall        = 0.597922

문제 LONG 유형 : BASE VS LONG-V2
station  layer            type   N  base_recall  v2_recall  long_prob_median  delta_recall
  S-ORS      2           drift 333     0.000000   0.000000          0.019561           0.0
  S-ORS      3          offset 121     0.000000   0.000000          0.004945           0.0
  I-ORS      1    drift+offset  45     0.000000   0.000000          0.001051           0.0
  I-ORS      1          offset 346     0.002890   0.002890          0.004070           0.0
  I-ORS      1           drift 339     0.070796   0.070796          0.000345           0.0
  S-ORS      8           drift

In [27]:
# =====================================================================
# LONG anomaly 전용 최소 피처
# 반드시 model_train / valid 분리 전에 train_df 전체에서 생성
# =====================================================================

print("Long-term drift/offset features 생성 중...")

train_df = train_df.sort_values(
    ['station', 'layer', 'time']
).copy()

group_cols = ['station', 'layer']


# ---------------------------------------------------------------------
# 1. 12h / 24h / 48h / 72h / 96h 과거 평균 대비 현재값 차이
# 10분 간격 기준
# ---------------------------------------------------------------------

windows = {
    '12h': 72,
    '24h': 144,
    '48h': 288,
    '72h': 432,
    '96h': 576
}


for name, window in windows.items():

    rolling_mean = (
        train_df
        .groupby(group_cols)['temp']
        .transform(
            lambda x:
            x.shift(1)
             .rolling(
                 window=window,
                 min_periods=max(12, window // 4)
             )
             .mean()
        )
    )

    train_df[
        f'temp_dev_{name}'
    ] = (
        train_df['temp']
        -
        rolling_mean
    )


# ---------------------------------------------------------------------
# 2. 장기 baseline 변화량
#
# 최근 12h 평균과 과거 48h 평균의 차이
# drift처럼 서서히 이동하는 현상을 노림
# ---------------------------------------------------------------------

recent_12h = (
    train_df
    .groupby(group_cols)['temp']
    .transform(
        lambda x:
        x.shift(1)
         .rolling(
             72,
             min_periods=36
         )
         .mean()
    )
)


past_48h = (
    train_df
    .groupby(group_cols)['temp']
    .transform(
        lambda x:
        x.shift(73)
         .rolling(
             288,
             min_periods=72
         )
         .mean()
    )
)


train_df[
    'long_baseline_shift'
] = (
    recent_12h
    -
    past_48h
)


# ---------------------------------------------------------------------
# 확인
# ---------------------------------------------------------------------

long_features = [
    'temp_dev_12h',
    'temp_dev_24h',
    'temp_dev_48h',
    'temp_dev_72h',
    'temp_dev_96h',
    'long_baseline_shift'
]


print(
    train_df[
        long_features
    ].describe().T
)

Long-term drift/offset features 생성 중...
                        count      mean       std        min       25%  \
temp_dev_12h         776418.0  0.003556  1.095995 -23.865458 -0.199135   
temp_dev_24h         776130.0  0.007234  1.165403 -23.655747 -0.228775   
temp_dev_48h         775554.0  0.014649  1.302305 -23.750658 -0.290590   
temp_dev_72h         774978.0  0.022395  1.397634 -23.825232 -0.351125   
temp_dev_96h         774402.0  0.029935  1.468657 -24.012276 -0.407904   
long_baseline_shift  774402.0  0.018852  0.974671 -15.099551 -0.223291   

                          50%       75%        max  
temp_dev_12h        -0.001852  0.214166  27.871607  
temp_dev_24h        -0.002563  0.239806  27.441681  
temp_dev_48h        -0.003830  0.298545  26.611279  
temp_dev_72h        -0.002387  0.358730  26.047156  
temp_dev_96h        -0.000612  0.416456  25.770001  
long_baseline_shift  0.008608  0.246966  14.034476  


In [22]:
long_model_features = (
    all_features
    +
    long_features
)

In [23]:
X_train_long = model_train[
    long_model_features
]

X_val_long = valid[
    long_model_features
]

In [29]:
# =====================================================================
# A. Train / Valid 다시 분할
# =====================================================================

model_train = train_df[
    (
        (train_df['time'] >= '2024-01-01')
        &
        (train_df['time'] <= '2024-12-31 23:59:59')
    )
    |
    (
        (train_df['time'] >= '2025-07-01')
        &
        (train_df['time'] <= '2025-12-31 23:59:59')
    )
].copy()


valid = train_df[
    (
        (train_df['time'] >= '2025-01-01')
        &
        (train_df['time'] <= '2025-06-30 23:59:59')
    )
].copy()


# =====================================================================
# B. 기존 BASE 150 Features
#
# 새 long feature는 BASE에는 넣지 않음
# =====================================================================

long_features = [
    'temp_dev_12h',
    'temp_dev_24h',
    'temp_dev_48h',
    'temp_dev_72h',
    'temp_dev_96h',
    'long_baseline_shift'
]

drop_cols = [
    'label',
    'anomaly_type',
    'time',
    'dataset',
    'depth',
    'depth_diff',
    'station',
    'layer'
] + long_features


all_features = [
    c
    for c in train_df.columns
    if c not in drop_cols
]


print(
    "BASE feature 개수:",
    len(all_features)
)


# =====================================================================
# C. BASE XGBoost 다시 학습
# =====================================================================

X_train = model_train[
    all_features
]

y_train = model_train[
    'label'
]

X_val = valid[
    all_features
]


scale_pos_weight = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)


xgb_model = XGBClassifier(

    objective='binary:logistic',

    n_estimators=1000,

    learning_rate=0.03,

    max_depth=6,

    min_child_weight=5,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    n_jobs=-1,

    eval_metric='logloss'
)


print("\nBASE XGBoost 재학습 중...")


xgb_model.fit(
    X_train,
    y_train
)


# =====================================================================
# D. Validation probability
# =====================================================================

valid['prob'] = (
    xgb_model
    .predict_proba(
        X_val
    )[:, 1]
)


# =====================================================================
# E. Global Threshold
# =====================================================================

global_scores = []


for th in np.arange(
    0.01,
    0.81,
    0.01
):

    pred = (
        valid['prob']
        >= th
    ).astype(int)

    f1 = f1_score(
        valid['label'],
        pred,
        zero_division=0
    )

    global_scores.append(
        (
            th,
            f1
        )
    )


GLOBAL_BEST_TH, GLOBAL_BEST_F1 = max(
    global_scores,
    key=lambda x: x[1]
)


print(
    f"\nGlobal TH = "
    f"{GLOBAL_BEST_TH:.2f}"
)

print(
    f"Global F1 = "
    f"{GLOBAL_BEST_F1:.6f}"
)


# =====================================================================
# F. Station × Layer Dynamic Threshold
# =====================================================================

best_thresholds = {}


for station in valid[
    'station'
].unique():

    layers = valid.loc[
        valid['station'] == station,
        'layer'
    ].unique()


    for layer in layers:

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )


        y_true_sl = valid.loc[
            mask,
            'label'
        ].values


        prob_sl = valid.loc[
            mask,
            'prob'
        ].values


        if y_true_sl.sum() == 0:

            best_thresholds[
                (station, layer)
            ] = GLOBAL_BEST_TH

            continue


        best_f1 = -1
        best_th = GLOBAL_BEST_TH


        for th in np.arange(
            0.01,
            0.81,
            0.01
        ):

            pred = (
                prob_sl >= th
            ).astype(int)


            f1 = f1_score(
                y_true_sl,
                pred,
                zero_division=0
            )


            if f1 > best_f1:

                best_f1 = f1
                best_th = th


        best_thresholds[
            (station, layer)
        ] = best_th


# =====================================================================
# G. base_pred 생성
# =====================================================================

valid['base_pred'] = 0


for station in valid[
    'station'
].unique():

    layers = valid.loc[
        valid['station'] == station,
        'layer'
    ].unique()


    for layer in layers:

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )


        th = best_thresholds[
            (station, layer)
        ]


        valid.loc[
            mask,
            'base_pred'
        ] = (
            valid.loc[
                mask,
                'prob'
            ]
            >= th
        ).astype(int)


print(
    "\nBase Dynamic F1:",
    f1_score(
        valid['label'],
        valid['base_pred'],
        zero_division=0
    )
)


# =====================================================================
# H. Speckle Only 후처리 함수
# =====================================================================

def apply_speckle_only(df):

    df_sorted = df.sort_values(
        'time'
    ).copy()


    s = df_sorted[
        'base_pred'
    ].astype(int).copy()


    is_one = pd.Series(
        s.values == 1,
        index=df_sorted.index
    )


    one_groups = pd.Series(
        s.values == 0,
        index=df_sorted.index
    ).cumsum()


    one_len = (
        is_one
        .groupby(one_groups)
        .transform('sum')
    )


    if 'abs_temp_diff_1' in df_sorted.columns:

        spike_in_group = (
            (
                df_sorted[
                    'abs_temp_diff_1'
                ]
                >= 1.7
            )
            .groupby(
                one_groups
            )
            .transform(
                'any'
            )
        )

    else:

        spike_in_group = pd.Series(
            False,
            index=df_sorted.index
        )


    invalid_block = (
        is_one
        &
        (one_len >= 2)
        &
        (one_len <= 11)
        &
        ~spike_in_group
    )


    s = pd.Series(
        np.where(
            invalid_block,
            0,
            s.values
        ),
        index=df_sorted.index
    )


    return s.sort_index()


# =====================================================================
# I. final_pred 생성
# =====================================================================

valid['final_pred'] = 0


for station in valid[
    'station'
].unique():

    layers = valid.loc[
        valid['station'] == station,
        'layer'
    ].unique()


    for layer in layers:

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )


        valid.loc[
            mask,
            'final_pred'
        ] = apply_speckle_only(
            valid.loc[
                mask
            ]
        )


BASE_FINAL_F1 = f1_score(
    valid['label'],
    valid['final_pred'],
    zero_division=0
)


print(
    "\nSpeckle Final F1:",
    BASE_FINAL_F1
)


# =====================================================================
# J. 이제 Long v3 target 생성
# =====================================================================

LONG_TYPES = [
    'drift',
    'offset',
    'drift+offset',
    'flatline+drift',
    'offset+flatline',
    'spike+drift'
]


model_train[
    'long_label'
] = (
    model_train[
        'anomaly_type'
    ]
    .isin(
        LONG_TYPES
    )
    .astype(int)
)


valid[
    'long_label'
] = (
    valid[
        'anomaly_type'
    ]
    .isin(
        LONG_TYPES
    )
    .astype(int)
)


# =====================================================================
# K. Long v3 Features
#
# 기존 150 + 신규 6개
# =====================================================================

long_model_features = (
    all_features
    +
    long_features
)


print(
    "\nLong v3 feature 개수:",
    len(long_model_features)
)


X_train_long = model_train[
    long_model_features
]

y_train_long = model_train[
    'long_label'
]

X_val_long = valid[
    long_model_features
]


# =====================================================================
# L. Long v3 학습
# =====================================================================

long_scale_pos_weight = (
    (y_train_long == 0).sum()
    /
    (y_train_long == 1).sum()
)


long_model_v3 = XGBClassifier(

    objective='binary:logistic',

    n_estimators=1000,

    learning_rate=0.03,

    max_depth=6,

    min_child_weight=5,

    subsample=0.8,

    colsample_bytree=0.8,

    scale_pos_weight=long_scale_pos_weight,

    random_state=42,

    n_jobs=-1,

    eval_metric='logloss'
)


print(
    "\nLong v3 학습 중..."
)


long_model_v3.fit(
    X_train_long,
    y_train_long
)


valid[
    'long_prob_v3'
] = (
    long_model_v3
    .predict_proba(
        X_val_long
    )[:, 1]
)


# =====================================================================
# M. Base + Long v3 Threshold 탐색
# =====================================================================

rows = []


base_pred = (
    valid[
        'final_pred'
    ]
    .astype(int)
    .values
)


y_true = (
    valid[
        'label'
    ]
    .astype(int)
    .values
)


for th in np.arange(
    0.01,
    0.96,
    0.01
):

    long_pred = (
        valid[
            'long_prob_v3'
        ].values
        >= th
    ).astype(int)


    combined = (
        (base_pred == 1)
        |
        (long_pred == 1)
    ).astype(int)


    rows.append({

        'threshold':
            th,

        'f1':
            f1_score(
                y_true,
                combined,
                zero_division=0
            ),

        'precision':
            precision_score(
                y_true,
                combined,
                zero_division=0
            ),

        'recall':
            recall_score(
                y_true,
                combined,
                zero_division=0
            ),

        'pred_ratio':
            combined.mean()
    })


v3_threshold_df = pd.DataFrame(
    rows
)


best_v3 = v3_threshold_df.loc[
    v3_threshold_df[
        'f1'
    ].idxmax()
]


BEST_LONG_V3_TH = best_v3[
    'threshold'
]


print("\n" + "=" * 100)

print(
    f"BASE F1        = "
    f"{BASE_FINAL_F1:.6f}"
)

print(
    f"LONG v3 TH     = "
    f"{BEST_LONG_V3_TH:.2f}"
)

print(
    f"COMBINED F1    = "
    f"{best_v3['f1']:.6f}"
)

print(
    f"DELTA F1       = "
    f"{best_v3['f1'] - BASE_FINAL_F1:+.6f}"
)

print(
    f"Precision      = "
    f"{best_v3['precision']:.6f}"
)

print(
    f"Recall         = "
    f"{best_v3['recall']:.6f}"
)

print("=" * 100)


# =====================================================================
# N. 최종 Long v3 combined_pred 생성
# =====================================================================

valid[
    'long_pred_v3'
] = (
    valid[
        'long_prob_v3'
    ]
    >= BEST_LONG_V3_TH
).astype(int)


valid[
    'combined_pred_v3'
] = (
    (
        valid[
            'final_pred'
        ] == 1
    )
    |
    (
        valid[
            'long_pred_v3'
        ] == 1
    )
).astype(int)


# =====================================================================
# O. 문제 Long anomaly probability 확인
# =====================================================================

print("\n" + "=" * 150)
print("LONG v3 : 문제 유형 Probability / Recall")
print("=" * 150)


rows = []


long_valid = valid[
    valid[
        'anomaly_type'
    ].isin(
        LONG_TYPES
    )
]


for station in sorted(
    long_valid[
        'station'
    ].unique()
):

    for layer in sorted(
        long_valid.loc[
            long_valid[
                'station'
            ] == station,
            'layer'
        ].unique()
    ):

        temp = long_valid[
            (
                long_valid[
                    'station'
                ] == station
            )
            &
            (
                long_valid[
                    'layer'
                ] == layer
            )
        ]


        for anomaly_type in sorted(
            temp[
                'anomaly_type'
            ].unique()
        ):

            x = temp[
                temp[
                    'anomaly_type'
                ]
                ==
                anomaly_type
            ]


            rows.append({

                'station':
                    station,

                'layer':
                    layer,

                'type':
                    anomaly_type,

                'N':
                    len(x),

                'base_recall':
                    x[
                        'final_pred'
                    ].mean(),

                'v3_recall':
                    x[
                        'combined_pred_v3'
                    ].mean(),

                'long_prob_median':
                    x[
                        'long_prob_v3'
                    ].median()
            })


v3_detail = pd.DataFrame(
    rows
)


v3_detail[
    'delta_recall'
] = (
    v3_detail[
        'v3_recall'
    ]
    -
    v3_detail[
        'base_recall'
    ]
)


print(
    v3_detail
    .sort_values(
        [
            'base_recall',
            'N'
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(
        index=False
    )
)

BASE feature 개수: 150

BASE XGBoost 재학습 중...

Global TH = 0.32
Global F1 = 0.581212

Base Dynamic F1: 0.6270376907901568

Speckle Final F1: 0.6345093420418652

Long v3 feature 개수: 156

Long v3 학습 중...

BASE F1        = 0.634509
LONG v3 TH     = 0.92
COMBINED F1    = 0.634739
DELTA F1       = +0.000230
Precision      = 0.675743
Recall         = 0.598426

LONG v3 : 문제 유형 Probability / Recall
station  layer            type   N  base_recall  v3_recall  long_prob_median  delta_recall
  S-ORS      2           drift 333     0.000000   0.000000          0.006750      0.000000
  S-ORS      3          offset 121     0.000000   0.000000          0.006852      0.000000
  I-ORS      1    drift+offset  45     0.000000   0.000000          0.001089      0.000000
  I-ORS      1          offset 346     0.002890   0.002890          0.002964      0.000000
  I-ORS      1           drift 339     0.070796   0.070796          0.000329      0.000000
  S-ORS      8           drift 614     0.140065   0.140065    

In [ ]:
# =====================================================================
# 31. 문제 Layer에서 기존 150개 Feature의 분리력 분석
# =====================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


# =====================================================================
# 1. 개별 Station × Layer × Anomaly Type 분석 함수
# =====================================================================

def analyze_feature_separation(
    df,
    station,
    layer,
    anomaly_type,
    features,
    min_valid_n=20
):

    # -------------------------------------------------------------
    # 해당 station × layer만 선택
    # -------------------------------------------------------------

    subset = df[
        (df['station'] == station)
        &
        (df['layer'] == layer)
    ].copy()


    # -------------------------------------------------------------
    # Normal vs 원하는 anomaly만 사용
    #
    # normal = label 0
    # anomaly = label 1 + 해당 anomaly_type
    # -------------------------------------------------------------

    normal = subset[
        subset['label'] == 0
    ].copy()


    anomaly = subset[
        (subset['label'] == 1)
        &
        (subset['anomaly_type'] == anomaly_type)
    ].copy()


    print("\n" + "=" * 110)

    print(
        f"{station} Layer {layer} | "
        f"{anomaly_type}"
    )

    print("=" * 110)

    print(
        f"Normal N  = {len(normal):,}"
    )

    print(
        f"Anomaly N = {len(anomaly):,}"
    )


    if len(anomaly) == 0:

        print("해당 anomaly가 없습니다.")

        return pd.DataFrame()


    results = []


    # =================================================================
    # Feature 하나씩 분석
    # =================================================================

    for feature in features:

        # 숫자형 아닌 경우 skip
        if not pd.api.types.is_numeric_dtype(
            subset[feature]
        ):
            continue


        normal_values = (
            normal[feature]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
        )


        anomaly_values = (
            anomaly[feature]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
        )


        if (
            len(normal_values) < min_valid_n
            or
            len(anomaly_values) < min_valid_n
        ):
            continue


        # -------------------------------------------------------------
        # 값이 사실상 constant면 skip
        # -------------------------------------------------------------

        combined_unique = pd.concat(
            [
                normal_values,
                anomaly_values
            ]
        ).nunique()


        if combined_unique <= 1:
            continue


        # -------------------------------------------------------------
        # ROC-AUC
        # -------------------------------------------------------------

        y = np.concatenate([
            np.zeros(
                len(normal_values)
            ),
            np.ones(
                len(anomaly_values)
            )
        ])


        x = np.concatenate([
            normal_values.values,
            anomaly_values.values
        ])


        try:

            auc_raw = roc_auc_score(
                y,
                x
            )

            # 방향과 무관한 분리력
            #
            # 0.9든 0.1이든 둘 다 잘 구분되는 feature
            auc_separation = max(
                auc_raw,
                1 - auc_raw
            )

        except ValueError:

            continue


        # -------------------------------------------------------------
        # 평균 / median
        # -------------------------------------------------------------

        normal_mean = (
            normal_values.mean()
        )

        anomaly_mean = (
            anomaly_values.mean()
        )


        normal_median = (
            normal_values.median()
        )

        anomaly_median = (
            anomaly_values.median()
        )


        # -------------------------------------------------------------
        # Cohen's d
        # -------------------------------------------------------------

        n0 = len(normal_values)
        n1 = len(anomaly_values)

        var0 = normal_values.var(
            ddof=1
        )

        var1 = anomaly_values.var(
            ddof=1
        )


        pooled_denominator = (
            n0 + n1 - 2
        )


        if (
            pooled_denominator > 0
            and
            np.isfinite(var0)
            and
            np.isfinite(var1)
        ):

            pooled_var = (
                (
                    (n0 - 1) * var0
                    +
                    (n1 - 1) * var1
                )
                /
                pooled_denominator
            )


            if (
                pooled_var > 0
                and
                np.isfinite(
                    pooled_var
                )
            ):

                cohen_d = (
                    anomaly_mean
                    -
                    normal_mean
                ) / np.sqrt(
                    pooled_var
                )

            else:

                cohen_d = np.nan

        else:

            cohen_d = np.nan


        # -------------------------------------------------------------
        # Robust separation:
        # median difference / normal IQR
        # -------------------------------------------------------------

        normal_q25 = (
            normal_values.quantile(
                0.25
            )
        )

        normal_q75 = (
            normal_values.quantile(
                0.75
            )
        )


        normal_iqr = (
            normal_q75
            -
            normal_q25
        )


        if (
            normal_iqr != 0
            and
            np.isfinite(
                normal_iqr
            )
        ):

            robust_diff = (
                anomaly_median
                -
                normal_median
            ) / normal_iqr

        else:

            robust_diff = np.nan


        # -------------------------------------------------------------
        # 결과 저장
        # -------------------------------------------------------------

        results.append({

            'station':
                station,

            'layer':
                layer,

            'anomaly_type':
                anomaly_type,

            'feature':
                feature,

            'normal_N':
                len(normal_values),

            'anomaly_N':
                len(anomaly_values),

            'auc_raw':
                auc_raw,

            'auc_sep':
                auc_separation,

            'normal_mean':
                normal_mean,

            'anomaly_mean':
                anomaly_mean,

            'normal_median':
                normal_median,

            'anomaly_median':
                anomaly_median,

            'median_diff':
                anomaly_median
                -
                normal_median,

            'cohen_d':
                cohen_d,

            'abs_cohen_d':
                (
                    abs(cohen_d)
                    if pd.notna(cohen_d)
                    else np.nan
                ),

            'robust_diff':
                robust_diff,

            'abs_robust_diff':
                (
                    abs(robust_diff)
                    if pd.notna(
                        robust_diff
                    )
                    else np.nan
                )
        })


    result_df = pd.DataFrame(
        results
    )


    if len(result_df) == 0:

        print(
            "분석 가능한 feature가 없습니다."
        )

        return result_df


    # =================================================================
    # AUC 기준 TOP 20
    # =================================================================

    print("\n[AUC 분리력 TOP 20]")

    print(
        result_df[
            [
                'feature',
                'auc_raw',
                'auc_sep',
                'normal_median',
                'anomaly_median',
                'cohen_d',
                'robust_diff'
            ]
        ]
        .sort_values(
            'auc_sep',
            ascending=False
        )
        .head(20)
        .to_string(
            index=False
        )
    )


    # =================================================================
    # 간단한 진단
    # =================================================================

    n_auc_60 = (
        result_df[
            'auc_sep'
        ] >= 0.60
    ).sum()


    n_auc_70 = (
        result_df[
            'auc_sep'
        ] >= 0.70
    ).sum()


    n_auc_80 = (
        result_df[
            'auc_sep'
        ] >= 0.80
    ).sum()


    print("\n분리력 요약")

    print(
        f"AUC ≥ 0.60 : "
        f"{n_auc_60}개"
    )

    print(
        f"AUC ≥ 0.70 : "
        f"{n_auc_70}개"
    )

    print(
        f"AUC ≥ 0.80 : "
        f"{n_auc_80}개"
    )

    print(
        f"최고 AUC    : "
        f"{result_df['auc_sep'].max():.4f}"
    )


    return result_df


# =====================================================================
# 2. 우리가 비교할 핵심 4개 Case
# =====================================================================

cases = [

    # --------------------------------------------
    # Drift
    # --------------------------------------------

    {
        'name':
            'S_L2_drift_BAD',

        'station':
            'S-ORS',

        'layer':
            2,

        'anomaly_type':
            'drift'
    },

    {
        'name':
            'S_L4_drift_GOOD',

        'station':
            'S-ORS',

        'layer':
            4,

        'anomaly_type':
            'drift'
    },


    # --------------------------------------------
    # Offset
    # --------------------------------------------

    {
        'name':
            'S_L3_offset_BAD',

        'station':
            'S-ORS',

        'layer':
            3,

        'anomaly_type':
            'offset'
    },

    {
        'name':
            'S_L5_offset_GOOD',

        'station':
            'S-ORS',

        'layer':
            5,

        'anomaly_type':
            'offset'
    }
]


# =====================================================================
# 3. 자동 분석
# =====================================================================

feature_analysis_results = {}


for case in cases:

    result = analyze_feature_separation(

        df=valid,

        station=case[
            'station'
        ],

        layer=case[
            'layer'
        ],

        anomaly_type=case[
            'anomaly_type'
        ],

        features=all_features
    )


    feature_analysis_results[
        case['name']
    ] = result


# =====================================================================
# 4. 네 Case를 한 표로 요약
# =====================================================================

summary_rows = []


for case_name, result in (
    feature_analysis_results.items()
):

    if len(result) == 0:
        continue


    summary_rows.append({

        'case':
            case_name,

        'n_features':
            len(result),

        'best_auc':
            result[
                'auc_sep'
            ].max(),

        'auc_60_count':
            (
                result[
                    'auc_sep'
                ] >= 0.60
            ).sum(),

        'auc_70_count':
            (
                result[
                    'auc_sep'
                ] >= 0.70
            ).sum(),

        'auc_80_count':
            (
                result[
                    'auc_sep'
                ] >= 0.80
            ).sum(),

        'best_feature':
            result.loc[
                result[
                    'auc_sep'
                ].idxmax(),
                'feature'
            ]
    })


feature_separation_summary = (
    pd.DataFrame(
        summary_rows
    )
)


print("\n" + "=" * 120)
print("4개 CASE Feature 분리력 비교")
print("=" * 120)


print(
    feature_separation_summary
    .to_string(
        index=False
    )
)


# =====================================================================
# 5. BAD vs GOOD에서 동일 Feature 비교
# =====================================================================

def compare_two_cases(
    result_bad,
    result_good,
    bad_name,
    good_name,
    top_n=30
):

    bad = result_bad[
        [
            'feature',
            'auc_sep',
            'abs_cohen_d',
            'abs_robust_diff'
        ]
    ].copy()


    good = result_good[
        [
            'feature',
            'auc_sep',
            'abs_cohen_d',
            'abs_robust_diff'
        ]
    ].copy()


    bad = bad.rename(
        columns={
            'auc_sep':
                f'{bad_name}_auc',

            'abs_cohen_d':
                f'{bad_name}_d',

            'abs_robust_diff':
                f'{bad_name}_robust'
        }
    )


    good = good.rename(
        columns={
            'auc_sep':
                f'{good_name}_auc',

            'abs_cohen_d':
                f'{good_name}_d',

            'abs_robust_diff':
                f'{good_name}_robust'
        }
    )


    merged = bad.merge(
        good,
        on='feature',
        how='inner'
    )


    merged[
        'auc_difference'
    ] = (
        merged[
            f'{good_name}_auc'
        ]
        -
        merged[
            f'{bad_name}_auc'
        ]
    )


    print("\n" + "=" * 130)

    print(
        f"{bad_name} VS {good_name}"
    )

    print("=" * 130)


    print(
        merged
        .sort_values(
            'auc_difference',
            ascending=False
        )
        .head(
            top_n
        )
        .to_string(
            index=False
        )
    )


    return merged


# =====================================================================
# 6. Drift BAD vs GOOD
# =====================================================================

drift_comparison = compare_two_cases(

    feature_analysis_results[
        'S_L2_drift_BAD'
    ],

    feature_analysis_results[
        'S_L4_drift_GOOD'
    ],

    bad_name='L2_drift',

    good_name='L4_drift'
)


# =====================================================================
# 7. Offset BAD vs GOOD
# =====================================================================

offset_comparison = compare_two_cases(

    feature_analysis_results[
        'S_L3_offset_BAD'
    ],

    feature_analysis_results[
        'S_L5_offset_GOOD'
    ],

    bad_name='L3_offset',

    good_name='L5_offset'
)









S-ORS Layer 2 | drift
Normal N  = 9,673
Anomaly N = 333

[AUC 분리력 TOP 20]
            feature  auc_raw  auc_sep  normal_median  anomaly_median   cohen_d  robust_diff
     temp_slope_144 0.088641 0.911359       0.001199       -0.007413 -1.421564    -1.554146
     temp_slope_288 0.127168 0.872832       0.001206       -0.007764 -1.715360    -3.129091
past_dev_median_576 0.148449 0.851551       0.342900       -1.366900 -1.503288    -1.692452
temp_dev_median_576 0.148583 0.851417       0.341750       -1.366900 -1.502328    -1.700657
       temp_std_288 0.848744 0.848744       0.648938        1.218192  1.016091     0.820491
       past_std_288 0.848673 0.848673       0.648938        1.218192  1.015203     0.820820
  past_residual_576 0.153705 0.846295       0.370895       -1.256961 -1.506255    -1.406036
  temp_residual_576 0.153762 0.846238       0.369641       -1.254990 -1.506179    -1.407990
    past_zscore_576 0.166435 0.833565       0.660645       -1.028399 -1.120743    -1.188773
    t

In [31]:
# =====================================================================
# TRAIN vs VALID
# Station × Layer × Anomaly Type 분포 비교
# =====================================================================

print("\n" + "=" * 150)
print("TRAIN vs VALID : STATION × LAYER × ANOMALY TYPE")
print("=" * 150)


# =====================================================================
# 1. Train anomaly count
# =====================================================================

train_counts = (
    model_train[
        model_train['label'] == 1
    ]
    .groupby(
        [
            'station',
            'layer',
            'anomaly_type'
        ]
    )
    .size()
    .reset_index(
        name='train_N'
    )
)


# =====================================================================
# 2. Validation anomaly count
# =====================================================================

valid_counts = (
    valid[
        valid['label'] == 1
    ]
    .groupby(
        [
            'station',
            'layer',
            'anomaly_type'
        ]
    )
    .size()
    .reset_index(
        name='valid_N'
    )
)


# =====================================================================
# 3. Merge
# =====================================================================

type_distribution = (
    train_counts
    .merge(
        valid_counts,
        on=[
            'station',
            'layer',
            'anomaly_type'
        ],
        how='outer'
    )
    .fillna(0)
)


type_distribution[
    ['train_N', 'valid_N']
] = (
    type_distribution[
        ['train_N', 'valid_N']
    ]
    .astype(int)
)


print(
    type_distribution
    .sort_values(
        [
            'station',
            'layer',
            'anomaly_type'
        ]
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 4. 지금 문제가 된 case만 따로 출력
# =====================================================================

problem_cases = [

    ('S-ORS', 2, 'drift'),

    ('S-ORS', 3, 'offset'),

    ('I-ORS', 1, 'drift'),

    ('I-ORS', 1, 'offset'),

    ('I-ORS', 1, 'drift+offset'),

    ('I-ORS', 7, 'drift'),

    ('I-ORS', 7, 'offset'),

    ('S-ORS', 8, 'drift'),

    ('S-ORS', 8, 'offset')
]


problem_rows = []


for station, layer, anomaly_type in problem_cases:

    row = type_distribution[
        (
            type_distribution['station']
            ==
            station
        )
        &
        (
            type_distribution['layer']
            ==
            layer
        )
        &
        (
            type_distribution['anomaly_type']
            ==
            anomaly_type
        )
    ]


    if len(row) == 0:

        train_n = 0
        valid_n = 0

    else:

        train_n = int(
            row.iloc[0][
                'train_N'
            ]
        )

        valid_n = int(
            row.iloc[0][
                'valid_N'
            ]
        )


    problem_rows.append({

        'station':
            station,

        'layer':
            layer,

        'anomaly_type':
            anomaly_type,

        'train_N':
            train_n,

        'valid_N':
            valid_n
    })


problem_distribution = pd.DataFrame(
    problem_rows
)


print("\n" + "=" * 100)
print("문제 CASE : TRAIN vs VALID")
print("=" * 100)

print(
    problem_distribution
    .to_string(
        index=False
    )
)


# =====================================================================
# 5. Train에 없는 Validation anomaly 찾기
# =====================================================================

unseen_cases = type_distribution[
    (
        type_distribution['train_N'] == 0
    )
    &
    (
        type_distribution['valid_N'] > 0
    )
].copy()


print("\n" + "=" * 120)
print("TRAIN에는 없는데 VALID에만 등장한 anomaly")
print("=" * 120)


if len(unseen_cases) == 0:

    print(
        "없음"
    )

else:

    print(
        unseen_cases
        .sort_values(
            'valid_N',
            ascending=False
        )
        .to_string(
            index=False
        )
    )


TRAIN vs VALID : STATION × LAYER × ANOMALY TYPE
station  layer      anomaly_type  train_N  valid_N
  G-ORS      1             drift      170        0
  G-ORS      1          flatline      385        0
  G-ORS      1             noise      251       68
  G-ORS      1       noise+drift       95        0
  G-ORS      1    noise+flatline       91        0
  G-ORS      1             spike        3        4
  I-ORS      1             drift      173      339
  I-ORS      1      drift+offset        0       45
  I-ORS      1          flatline        0      241
  I-ORS      1             noise      120        0
  I-ORS      1            offset      363      346
  I-ORS      1             spike        2        2
  I-ORS      2             drift      104        0
  I-ORS      2          flatline      301       43
  I-ORS      2    flatline+spike        1        0
  I-ORS      2             noise       28        0
  I-ORS      2            offset      268      221
  I-ORS      2      offset+drift 

In [32]:
# =====================================================================
# 32. TYPE-SPECIFIC MODEL
#
# Drift 전용 / Offset 전용
#
# 핵심:
# 다른 anomaly를 negative로 취급하지 않음
#
# Drift model  : Normal vs Drift-containing anomaly
# Offset model : Normal vs Offset-containing anomaly
# =====================================================================

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 0. 현재 BASE 확인
# =====================================================================

assert 'final_pred' in valid.columns

BASE_F1 = f1_score(
    valid['label'],
    valid['final_pred'],
    zero_division=0
)

print("=" * 110)
print("TYPE-SPECIFIC MODEL TEST")
print("=" * 110)

print(
    f"현재 BASE F1 = "
    f"{BASE_F1:.6f}"
)


# =====================================================================
# 1. anomaly_type 문자열 준비
# =====================================================================

train_type = (
    model_train['anomaly_type']
    .fillna('')
    .astype(str)
)

valid_type = (
    valid['anomaly_type']
    .fillna('')
    .astype(str)
)


# =====================================================================
# 2. TYPE 전용 모델 학습 함수
# =====================================================================

def train_type_model(
    type_keyword,
    model_name
):

    # -------------------------------------------------------------
    # Positive
    # 해당 keyword 포함 anomaly
    # -------------------------------------------------------------

    positive_mask = (
        (model_train['label'] == 1)
        &
        train_type.str.contains(
            type_keyword,
            regex=False
        )
    )


    # -------------------------------------------------------------
    # Negative
    # 진짜 NORMAL만 사용
    #
    # 다른 anomaly는 학습에서 제외
    # -------------------------------------------------------------

    normal_mask = (
        model_train['label'] == 0
    )


    train_mask = (
        positive_mask
        |
        normal_mask
    )


    X_type_train = model_train.loc[
        train_mask,
        all_features
    ]


    y_type_train = (
        positive_mask.loc[
            train_mask
        ]
        .astype(int)
    )


    # -------------------------------------------------------------
    # Class weight
    # -------------------------------------------------------------

    scale = (
        (y_type_train == 0).sum()
        /
        (y_type_train == 1).sum()
    )


    print("\n" + "=" * 100)

    print(
        f"{model_name} MODEL"
    )

    print("=" * 100)

    print(
        f"Normal train = "
        f"{(y_type_train == 0).sum():,}"
    )

    print(
        f"Positive train = "
        f"{(y_type_train == 1).sum():,}"
    )

    print(
        f"scale_pos_weight = "
        f"{scale:.3f}"
    )


    # -------------------------------------------------------------
    # Model
    # -------------------------------------------------------------

    model = XGBClassifier(

        objective='binary:logistic',

        n_estimators=1000,

        learning_rate=0.03,

        max_depth=6,

        min_child_weight=5,

        subsample=0.8,

        colsample_bytree=0.8,

        scale_pos_weight=scale,

        random_state=42,

        n_jobs=-1,

        eval_metric='logloss'
    )


    print(
        f"{model_name} XGBoost 학습 중..."
    )


    model.fit(
        X_type_train,
        y_type_train
    )


    # -------------------------------------------------------------
    # Validation probability
    # -------------------------------------------------------------

    prob = model.predict_proba(
        valid[all_features]
    )[:, 1]


    return model, prob


# =====================================================================
# 3. DRIFT 모델
# =====================================================================

drift_model, drift_prob = train_type_model(
    type_keyword='drift',
    model_name='DRIFT'
)


valid[
    'drift_prob'
] = drift_prob


# =====================================================================
# 4. OFFSET 모델
# =====================================================================

offset_model, offset_prob = train_type_model(
    type_keyword='offset',
    model_name='OFFSET'
)


valid[
    'offset_prob'
] = offset_prob


# =====================================================================
# 5. BASE + DRIFT threshold 탐색
#
# Type 모델 자체 F1이 아니라
# 최종 전체 label F1을 기준으로 threshold 선택
# =====================================================================

base_pred = (
    valid['final_pred']
    .astype(int)
    .values
)

y_true = (
    valid['label']
    .astype(int)
    .values
)


drift_threshold_rows = []


for th in np.arange(
    0.01,
    0.96,
    0.01
):

    drift_pred = (
        valid['drift_prob'].values
        >= th
    ).astype(int)


    combined = (
        (base_pred == 1)
        |
        (drift_pred == 1)
    ).astype(int)


    # 실제 drift가 포함된 validation anomaly
    drift_valid_mask = (
        (valid['label'] == 1)
        &
        valid_type.str.contains(
            'drift',
            regex=False
        )
    ).values


    if drift_valid_mask.sum() > 0:

        drift_recall = (
            combined[
                drift_valid_mask
            ].mean()
        )

    else:

        drift_recall = np.nan


    drift_threshold_rows.append({

        'threshold':
            th,

        'overall_f1':
            f1_score(
                y_true,
                combined,
                zero_division=0
            ),

        'precision':
            precision_score(
                y_true,
                combined,
                zero_division=0
            ),

        'recall':
            recall_score(
                y_true,
                combined,
                zero_division=0
            ),

        'drift_recall':
            drift_recall,

        'pred_ratio':
            combined.mean()
    })


drift_threshold_df = pd.DataFrame(
    drift_threshold_rows
)


best_drift = drift_threshold_df.loc[
    drift_threshold_df[
        'overall_f1'
    ].idxmax()
]


BEST_DRIFT_TH = (
    best_drift[
        'threshold'
    ]
)


print("\n" + "=" * 110)
print("BASE + DRIFT")
print("=" * 110)

print(
    f"Base F1        = "
    f"{BASE_F1:.6f}"
)

print(
    f"Best Drift TH  = "
    f"{BEST_DRIFT_TH:.2f}"
)

print(
    f"Combined F1    = "
    f"{best_drift['overall_f1']:.6f}"
)

print(
    f"Delta F1       = "
    f"{best_drift['overall_f1'] - BASE_F1:+.6f}"
)

print(
    f"Drift Recall   = "
    f"{best_drift['drift_recall']:.6f}"
)


# =====================================================================
# 6. DRIFT 적용
# =====================================================================

valid[
    'drift_pred'
] = (
    valid[
        'drift_prob'
    ]
    >= BEST_DRIFT_TH
).astype(int)


valid[
    'base_plus_drift'
] = (
    (
        valid[
            'final_pred'
        ] == 1
    )
    |
    (
        valid[
            'drift_pred'
        ] == 1
    )
).astype(int)


# =====================================================================
# 7. BASE + DRIFT + OFFSET threshold 탐색
#
# Drift threshold는 고정
# Offset만 추가 탐색
# =====================================================================

drift_base = (
    valid[
        'base_plus_drift'
    ]
    .astype(int)
    .values
)


offset_threshold_rows = []


for th in np.arange(
    0.01,
    0.96,
    0.01
):

    offset_pred = (
        valid[
            'offset_prob'
        ].values
        >= th
    ).astype(int)


    combined = (
        (drift_base == 1)
        |
        (offset_pred == 1)
    ).astype(int)


    offset_valid_mask = (
        (valid['label'] == 1)
        &
        valid_type.str.contains(
            'offset',
            regex=False
        )
    ).values


    if offset_valid_mask.sum() > 0:

        offset_recall = (
            combined[
                offset_valid_mask
            ].mean()
        )

    else:

        offset_recall = np.nan


    offset_threshold_rows.append({

        'threshold':
            th,

        'overall_f1':
            f1_score(
                y_true,
                combined,
                zero_division=0
            ),

        'precision':
            precision_score(
                y_true,
                combined,
                zero_division=0
            ),

        'recall':
            recall_score(
                y_true,
                combined,
                zero_division=0
            ),

        'offset_recall':
            offset_recall,

        'pred_ratio':
            combined.mean()
    })


offset_threshold_df = pd.DataFrame(
    offset_threshold_rows
)


best_offset = offset_threshold_df.loc[
    offset_threshold_df[
        'overall_f1'
    ].idxmax()
]


BEST_OFFSET_TH = (
    best_offset[
        'threshold'
    ]
)


print("\n" + "=" * 110)
print("BASE + DRIFT + OFFSET")
print("=" * 110)

print(
    f"BASE F1         = "
    f"{BASE_F1:.6f}"
)

print(
    f"Drift TH        = "
    f"{BEST_DRIFT_TH:.2f}"
)

print(
    f"Offset TH       = "
    f"{BEST_OFFSET_TH:.2f}"
)

print(
    f"Final F1        = "
    f"{best_offset['overall_f1']:.6f}"
)

print(
    f"Delta F1        = "
    f"{best_offset['overall_f1'] - BASE_F1:+.6f}"
)

print(
    f"Precision       = "
    f"{best_offset['precision']:.6f}"
)

print(
    f"Recall          = "
    f"{best_offset['recall']:.6f}"
)

print(
    f"Offset Recall   = "
    f"{best_offset['offset_recall']:.6f}"
)


# =====================================================================
# 8. 최종 Type-specific prediction
# =====================================================================

valid[
    'offset_pred'
] = (
    valid[
        'offset_prob'
    ]
    >= BEST_OFFSET_TH
).astype(int)


valid[
    'type_combined_pred'
] = (
    (
        valid[
            'base_plus_drift'
        ] == 1
    )
    |
    (
        valid[
            'offset_pred'
        ] == 1
    )
).astype(int)


# =====================================================================
# 9. 문제 CASE에서 probability / recall 변화 확인
# =====================================================================

problem_cases = [

    ('S-ORS', 2, 'drift'),

    ('S-ORS', 3, 'offset'),

    ('I-ORS', 1, 'drift'),

    ('I-ORS', 1, 'offset'),

    ('I-ORS', 1, 'drift+offset'),

    ('I-ORS', 7, 'drift'),

    ('I-ORS', 7, 'offset'),

    ('S-ORS', 8, 'drift'),

    ('S-ORS', 8, 'offset')
]


problem_results = []


for station, layer, anomaly_type in problem_cases:

    mask = (
        (valid['station'] == station)
        &
        (valid['layer'] == layer)
        &
        (valid['anomaly_type'] == anomaly_type)
    )


    temp = valid.loc[
        mask
    ]


    if len(temp) == 0:
        continue


    problem_results.append({

        'station':
            station,

        'layer':
            layer,

        'type':
            anomaly_type,

        'N':
            len(temp),

        'base_recall':
            temp[
                'final_pred'
            ].mean(),

        'new_recall':
            temp[
                'type_combined_pred'
            ].mean(),

        'drift_prob_median':
            temp[
                'drift_prob'
            ].median(),

        'offset_prob_median':
            temp[
                'offset_prob'
            ].median()
    })


problem_type_result = pd.DataFrame(
    problem_results
)


problem_type_result[
    'delta_recall'
] = (
    problem_type_result[
        'new_recall'
    ]
    -
    problem_type_result[
        'base_recall'
    ]
)


print("\n" + "=" * 150)
print("문제 CASE : TYPE-SPECIFIC 모델 결과")
print("=" * 150)

print(
    problem_type_result
    .to_string(
        index=False
    )
)


# =====================================================================
# 10. 상위 Threshold 확인
# =====================================================================

print("\nDRIFT Threshold TOP 10")

print(
    drift_threshold_df
    .sort_values(
        'overall_f1',
        ascending=False
    )
    .head(10)
    .to_string(
        index=False
    )
)


print("\nOFFSET Threshold TOP 10")

print(
    offset_threshold_df
    .sort_values(
        'overall_f1',
        ascending=False
    )
    .head(10)
    .to_string(
        index=False
    )
)

TYPE-SPECIFIC MODEL TEST
현재 BASE F1 = 0.634509

DRIFT MODEL
Normal train = 546,398
Positive train = 5,379
scale_pos_weight = 101.580
DRIFT XGBoost 학습 중...

OFFSET MODEL
Normal train = 546,398
Positive train = 4,715
scale_pos_weight = 115.885
OFFSET XGBoost 학습 중...

BASE + DRIFT
Base F1        = 0.634509
Best Drift TH  = 0.29
Combined F1    = 0.635890
Delta F1       = +0.001381
Drift Recall   = 0.349859

BASE + DRIFT + OFFSET
BASE F1         = 0.634509
Drift TH        = 0.29
Offset TH       = 0.95
Final F1        = 0.635789
Delta F1        = +0.001280
Precision       = 0.671265
Recall          = 0.603874
Offset Recall   = 0.457736

문제 CASE : TYPE-SPECIFIC 모델 결과
station  layer         type   N  base_recall  new_recall  drift_prob_median  offset_prob_median  delta_recall
  S-ORS      2        drift 333     0.000000    0.078078           0.009951            0.000004      0.078078
  S-ORS      3       offset 121     0.000000    0.000000           0.000220            0.000190      0.000000
 

In [ ]:
# =====================================================================
# 33. NORMAL-ONLY ROBUST DETECTOR
#     - LONG_DETECT_FEATURES 4개만 사용
#     - Train의 NORMAL 분포만 학습
# =====================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 0. 사용할 feature 딱 4개
# =====================================================================

LONG_DETECT_FEATURES = [
    'temp_slope_144',
    'temp_slope_288',
    'temp_dev_median_288',
    'temp_dev_median_576'
]


# =====================================================================
# 1. 현재 BASE 확인
# =====================================================================

assert 'final_pred' in valid.columns, \
    "valid['final_pred']가 없습니다. Base + Speckle 결과를 먼저 생성하세요."


for feature in LONG_DETECT_FEATURES:

    assert feature in model_train.columns, \
        f"{feature}가 model_train에 없습니다."

    assert feature in valid.columns, \
        f"{feature}가 valid에 없습니다."


BASE_F1 = f1_score(
    valid['label'],
    valid['final_pred'],
    zero_division=0
)


print("=" * 120)
print("NORMAL-ONLY ROBUST DETECTOR")
print("=" * 120)

print(
    f"현재 BASE F1 = "
    f"{BASE_F1:.6f}"
)

print(
    "\n사용 Features:"
)

for f in LONG_DETECT_FEATURES:
    print(" -", f)


# =====================================================================
# 2. Train NORMAL 데이터만 사용
# =====================================================================

normal_train = model_train[
    model_train['label'] == 0
].copy()


print(
    f"\nTrain Normal N = "
    f"{len(normal_train):,}"
)


# =====================================================================
# 3. Station × Layer별 Normal median / IQR 계산
# =====================================================================

normal_stats_rows = []


for station in sorted(
    model_train['station'].unique()
):

    layers = sorted(
        model_train.loc[
            model_train['station'] == station,
            'layer'
        ].unique()
    )


    for layer in layers:

        mask = (
            (normal_train['station'] == station)
            &
            (normal_train['layer'] == layer)
        )

        temp = normal_train.loc[
            mask
        ]


        if len(temp) == 0:
            continue


        for feature in LONG_DETECT_FEATURES:

            values = (
                temp[feature]
                .replace(
                    [np.inf, -np.inf],
                    np.nan
                )
                .dropna()
            )


            if len(values) == 0:
                continue


            median = (
                values.median()
            )

            q25 = (
                values.quantile(
                    0.25
                )
            )

            q75 = (
                values.quantile(
                    0.75
                )
            )

            iqr = (
                q75 - q25
            )


            normal_stats_rows.append({

                'station':
                    station,

                'layer':
                    layer,

                'feature':
                    feature,

                'N':
                    len(values),

                'median':
                    median,

                'q25':
                    q25,

                'q75':
                    q75,

                'iqr':
                    iqr
            })


normal_stats = pd.DataFrame(
    normal_stats_rows
)


print("\nNormal statistics sample:")

print(
    normal_stats.head(
        20
    ).to_string(
        index=False
    )
)


# =====================================================================
# 4. Validation에 Robust Z-score 계산
#
# robust_z = (x - normal_median) / IQR
# =====================================================================

valid_robust = valid.copy()


for feature in LONG_DETECT_FEATURES:

    valid_robust[
        f'robust_z_{feature}'
    ] = np.nan


# =====================================================================
# 5. station × layer별 정상분포를 Valid에 적용
# =====================================================================

for station in valid_robust[
    'station'
].unique():

    for layer in valid_robust.loc[
        valid_robust['station'] == station,
        'layer'
    ].unique():

        mask = (
            (valid_robust['station'] == station)
            &
            (valid_robust['layer'] == layer)
        )


        stats_sl = normal_stats[
            (normal_stats['station'] == station)
            &
            (normal_stats['layer'] == layer)
        ]


        # -------------------------------------------------------------
        # 해당 station-layer가 Train에 없는 경우
        # station 전체 normal stats로 fallback
        # -------------------------------------------------------------

        if len(stats_sl) == 0:

            stats_sl = normal_stats[
                normal_stats[
                    'station'
                ] == station
            ]


        # -------------------------------------------------------------
        # 그것도 없으면 global fallback
        # -------------------------------------------------------------

        if len(stats_sl) == 0:

            stats_sl = normal_stats


        for feature in LONG_DETECT_FEATURES:

            feature_stat = stats_sl[
                stats_sl['feature'] == feature
            ]


            if len(feature_stat) == 0:
                continue


            # 여러 layer가 fallback으로 들어온 경우 median 사용
            normal_median = (
                feature_stat[
                    'median'
                ].median()
            )

            normal_iqr = (
                feature_stat[
                    'iqr'
                ].median()
            )


            # IQR 0 방어
            if (
                pd.isna(normal_iqr)
                or
                normal_iqr == 0
            ):

                normal_iqr = 1e-8


            valid_robust.loc[
                mask,
                f'robust_z_{feature}'
            ] = (
                valid_robust.loc[
                    mask,
                    feature
                ]
                -
                normal_median
            ) / normal_iqr


# =====================================================================
# 6. 4개 feature의 |robust z| 생성
# =====================================================================

robust_z_cols = [

    f'robust_z_{feature}'
    for feature in LONG_DETECT_FEATURES
]


for col in robust_z_cols:

    valid_robust[
        f'abs_{col}'
    ] = (
        valid_robust[
            col
        ].abs()
    )


abs_robust_cols = [

    f'abs_robust_z_{feature}'
    for feature in LONG_DETECT_FEATURES
]


# =====================================================================
# 7. Normal-only anomaly score
#
# 가장 크게 정상범위를 벗어난 feature를 사용
# =====================================================================

valid_robust[
    'normal_only_score'
] = (
    valid_robust[
        abs_robust_cols
    ]
    .max(
        axis=1,
        skipna=True
    )
)


print("\n" + "=" * 100)
print("NORMAL-ONLY SCORE 분포")
print("=" * 100)

print(
    valid_robust[
        'normal_only_score'
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
            0.995
        ]
    )
)


# =====================================================================
# 8. Threshold 탐색
#
# Base OR Normal-only detector
# 최종 전체 F1 기준
# =====================================================================

base_pred = (
    valid_robust[
        'final_pred'
    ]
    .astype(int)
    .values
)


y_true = (
    valid_robust[
        'label'
    ]
    .astype(int)
    .values
)


threshold_rows = []


# robust z-score이므로 0.5 ~ 10 정도 탐색
for th in np.arange(
    0.5,
    10.01,
    0.1
):

    normal_pred = (
        valid_robust[
            'normal_only_score'
        ].values
        >= th
    ).astype(int)


    combined = (
        (base_pred == 1)
        |
        (normal_pred == 1)
    ).astype(int)


    # -------------------------------------------------------------
    # Drift recall
    # -------------------------------------------------------------

    drift_mask = (
        (valid_robust['label'] == 1)
        &
        valid_robust[
            'anomaly_type'
        ]
        .fillna('')
        .astype(str)
        .str.contains(
            'drift',
            regex=False
        )
    ).values


    if drift_mask.sum() > 0:

        drift_recall = (
            combined[
                drift_mask
            ].mean()
        )

    else:

        drift_recall = np.nan


    # -------------------------------------------------------------
    # Offset recall
    # -------------------------------------------------------------

    offset_mask = (
        (valid_robust['label'] == 1)
        &
        valid_robust[
            'anomaly_type'
        ]
        .fillna('')
        .astype(str)
        .str.contains(
            'offset',
            regex=False
        )
    ).values


    if offset_mask.sum() > 0:

        offset_recall = (
            combined[
                offset_mask
            ].mean()
        )

    else:

        offset_recall = np.nan


    threshold_rows.append({

        'threshold':
            th,

        'overall_f1':
            f1_score(
                y_true,
                combined,
                zero_division=0
            ),

        'precision':
            precision_score(
                y_true,
                combined,
                zero_division=0
            ),

        'recall':
            recall_score(
                y_true,
                combined,
                zero_division=0
            ),

        'drift_recall':
            drift_recall,

        'offset_recall':
            offset_recall,

        'pred_ratio':
            combined.mean(),

        'normal_only_ratio':
            normal_pred.mean()
    })


normal_only_threshold_df = pd.DataFrame(
    threshold_rows
)


# =====================================================================
# 9. Best Threshold
# =====================================================================

best_idx = (
    normal_only_threshold_df[
        'overall_f1'
    ].idxmax()
)


best_normal_only = (
    normal_only_threshold_df.loc[
        best_idx
    ]
)


BEST_NORMAL_ONLY_TH = (
    best_normal_only[
        'threshold'
    ]
)


print("\n" + "=" * 120)
print("NORMAL-ONLY DETECTOR 결과")
print("=" * 120)

print(
    f"BASE F1             = "
    f"{BASE_F1:.6f}"
)

print(
    f"Best Robust TH      = "
    f"{BEST_NORMAL_ONLY_TH:.2f}"
)

print(
    f"Combined F1         = "
    f"{best_normal_only['overall_f1']:.6f}"
)

print(
    f"Delta F1            = "
    f"{best_normal_only['overall_f1'] - BASE_F1:+.6f}"
)

print(
    f"Precision           = "
    f"{best_normal_only['precision']:.6f}"
)

print(
    f"Recall              = "
    f"{best_normal_only['recall']:.6f}"
)

print(
    f"Drift Recall        = "
    f"{best_normal_only['drift_recall']:.6f}"
)

print(
    f"Offset Recall       = "
    f"{best_normal_only['offset_recall']:.6f}"
)

print(
    f"Prediction Ratio    = "
    f"{best_normal_only['pred_ratio']:.4%}"
)


# =====================================================================
# 10. Best threshold로 최종 prediction
# =====================================================================

valid_robust[
    'normal_only_pred'
] = (
    valid_robust[
        'normal_only_score'
    ]
    >= BEST_NORMAL_ONLY_TH
).astype(int)


valid_robust[
    'combined_normal_pred'
] = (
    (
        valid_robust[
            'final_pred'
        ] == 1
    )
    |
    (
        valid_robust[
            'normal_only_pred'
        ] == 1
    )
).astype(int)


# =====================================================================
# 11. 문제 CASE Recall 비교
# =====================================================================

problem_cases = [

    ('S-ORS', 2, 'drift'),

    ('S-ORS', 3, 'offset'),

    ('I-ORS', 1, 'drift'),

    ('I-ORS', 1, 'offset'),

    ('I-ORS', 1, 'drift+offset'),

    ('I-ORS', 7, 'drift'),

    ('I-ORS', 7, 'offset'),

    ('S-ORS', 8, 'drift'),

    ('S-ORS', 8, 'offset')
]


problem_rows = []


for station, layer, anomaly_type in problem_cases:

    mask = (
        (valid_robust['station'] == station)
        &
        (valid_robust['layer'] == layer)
        &
        (
            valid_robust[
                'anomaly_type'
            ] == anomaly_type
        )
    )


    temp = valid_robust.loc[
        mask
    ]


    if len(temp) == 0:
        continue


    problem_rows.append({

        'station':
            station,

        'layer':
            layer,

        'anomaly_type':
            anomaly_type,

        'N':
            len(temp),

        'base_recall':
            temp[
                'final_pred'
            ].mean(),

        'normal_detector_recall':
            temp[
                'combined_normal_pred'
            ].mean(),

        'delta_recall':
            (
                temp[
                    'combined_normal_pred'
                ].mean()
                -
                temp[
                    'final_pred'
                ].mean()
            ),

        'score_median':
            temp[
                'normal_only_score'
            ].median(),

        'score_q75':
            temp[
                'normal_only_score'
            ].quantile(
                0.75
            ),

        'score_q90':
            temp[
                'normal_only_score'
            ].quantile(
                0.90
            )
    })


problem_normal_result = pd.DataFrame(
    problem_rows
)


print("\n" + "=" * 160)
print("문제 CASE : NORMAL-ONLY Detector Recall")
print("=" * 160)


print(
    problem_normal_result
    .to_string(
        index=False
    )
)


# =====================================================================
# 12. Station별 F1 비교
# =====================================================================

station_rows = []


for station in sorted(
    valid_robust[
        'station'
    ].unique()
):

    mask = (
        valid_robust[
            'station'
        ] == station
    )


    y_station = valid_robust.loc[
        mask,
        'label'
    ]


    base_station = valid_robust.loc[
        mask,
        'final_pred'
    ]


    new_station = valid_robust.loc[
        mask,
        'combined_normal_pred'
    ]


    station_rows.append({

        'station':
            station,

        'base_f1':
            f1_score(
                y_station,
                base_station,
                zero_division=0
            ),

        'normal_combined_f1':
            f1_score(
                y_station,
                new_station,
                zero_division=0
            ),

        'delta_f1':
            (
                f1_score(
                    y_station,
                    new_station,
                    zero_division=0
                )
                -
                f1_score(
                    y_station,
                    base_station,
                    zero_division=0
                )
            ),

        'new_precision':
            precision_score(
                y_station,
                new_station,
                zero_division=0
            ),

        'new_recall':
            recall_score(
                y_station,
                new_station,
                zero_division=0
            )
    })


station_normal_compare = pd.DataFrame(
    station_rows
)


print("\n" + "=" * 120)
print("STATION별 BASE VS NORMAL-ONLY")
print("=" * 120)


print(
    station_normal_compare
    .to_string(
        index=False
    )
)


# =====================================================================
# 13. Threshold TOP 15
# =====================================================================

print("\n" + "=" * 140)
print("NORMAL-ONLY Threshold TOP 15")
print("=" * 140)


print(
    normal_only_threshold_df
    .sort_values(
        'overall_f1',
        ascending=False
    )
    .head(
        15
    )
    .to_string(
        index=False
    )
)



NORMAL-ONLY ROBUST DETECTOR
현재 BASE F1 = 0.634509

사용 Features:
 - temp_slope_144
 - temp_slope_288
 - temp_dev_median_288
 - temp_dev_median_576

Train Normal N = 546,398

Normal statistics sample:
station  layer             feature     N    median       q25      q75      iqr
  G-ORS      1      temp_slope_144  8578 -0.000092 -0.001754 0.001836 0.003590
  G-ORS      1      temp_slope_288  8578  0.000185 -0.000992 0.001325 0.002317
  G-ORS      1 temp_dev_median_288  8578  0.008425 -0.194125 0.215850 0.409975
  G-ORS      1 temp_dev_median_576  8578  0.063625 -0.256312 0.342200 0.598512
  I-ORS      1      temp_slope_144 19268 -0.000247 -0.002132 0.001640 0.003772
  I-ORS      1      temp_slope_288 19268 -0.000229 -0.001290 0.001005 0.002295
  I-ORS      1 temp_dev_median_288 19268 -0.052625 -0.267512 0.172463 0.439975
  I-ORS      1 temp_dev_median_576 19268 -0.081725 -0.417413 0.243462 0.660875
  I-ORS      2      temp_slope_144 19197 -0.000182 -0.001987 0.001638 0.003625
  I-ORS    

In [ ]:
# =====================================================================
# 34. ROBUST SCORE AGGREGATION / VOTING 비교
# =====================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 0. 사용할 4개 robust z-score
# =====================================================================

ROBUST_Z_COLS = [
    'robust_z_temp_slope_144',
    'robust_z_temp_slope_288',
    'robust_z_temp_dev_median_288',
    'robust_z_temp_dev_median_576'
]

ABS_Z_COLS = []

for col in ROBUST_Z_COLS:

    abs_col = f'abs_{col}'

    valid_robust[abs_col] = (
        valid_robust[col].abs()
    )

    ABS_Z_COLS.append(
        abs_col
    )


# =====================================================================
# 1. 여러 aggregation score 생성
# =====================================================================

valid_robust[
    'score_max'
] = (
    valid_robust[
        ABS_Z_COLS
    ]
    .max(axis=1)
)


valid_robust[
    'score_mean'
] = (
    valid_robust[
        ABS_Z_COLS
    ]
    .mean(axis=1)
)


valid_robust[
    'score_median'
] = (
    valid_robust[
        ABS_Z_COLS
    ]
    .median(axis=1)
)


# =====================================================================
# 2. Base
# =====================================================================

base_pred = (
    valid_robust[
        'final_pred'
    ]
    .astype(int)
    .values
)

y_true = (
    valid_robust[
        'label'
    ]
    .astype(int)
    .values
)


BASE_F1 = f1_score(
    y_true,
    base_pred,
    zero_division=0
)


print("=" * 120)
print("ROBUST AGGREGATION / VOTING TEST")
print("=" * 120)

print(
    f"BASE F1 = "
    f"{BASE_F1:.6f}"
)


# =====================================================================
# 3. Drift / Offset mask
# =====================================================================

anomaly_string = (
    valid_robust[
        'anomaly_type'
    ]
    .fillna('')
    .astype(str)
)


drift_mask = (
    (valid_robust['label'] == 1)
    &
    anomaly_string.str.contains(
        'drift',
        regex=False
    )
).values


offset_mask = (
    (valid_robust['label'] == 1)
    &
    anomaly_string.str.contains(
        'offset',
        regex=False
    )
).values


# =====================================================================
# 4. Continuous score 탐색 함수
# =====================================================================

def test_continuous_score(
    score_col,
    score_name,
    thresholds
):

    rows = []


    for th in thresholds:

        detector_pred = (
            valid_robust[
                score_col
            ].values
            >= th
        ).astype(int)


        combined = (
            (base_pred == 1)
            |
            (detector_pred == 1)
        ).astype(int)


        rows.append({

            'method':
                score_name,

            'threshold':
                th,

            'vote_k':
                np.nan,

            'overall_f1':
                f1_score(
                    y_true,
                    combined,
                    zero_division=0
                ),

            'precision':
                precision_score(
                    y_true,
                    combined,
                    zero_division=0
                ),

            'recall':
                recall_score(
                    y_true,
                    combined,
                    zero_division=0
                ),

            'drift_recall':
                combined[
                    drift_mask
                ].mean(),

            'offset_recall':
                combined[
                    offset_mask
                ].mean(),

            'pred_ratio':
                combined.mean(),

            'detector_ratio':
                detector_pred.mean()
        })


    return rows


# =====================================================================
# 5. MAX / MEAN / MEDIAN 비교
# =====================================================================

all_rows = []


score_thresholds = np.arange(
    0.5,
    10.01,
    0.1
)


all_rows += test_continuous_score(
    'score_max',
    'MAX',
    score_thresholds
)


all_rows += test_continuous_score(
    'score_mean',
    'MEAN',
    score_thresholds
)


all_rows += test_continuous_score(
    'score_median',
    'MEDIAN',
    score_thresholds
)


# =====================================================================
# 6. Voting
#
# z threshold를 넘는 feature 개수가
# 최소 K개이면 detector=1
#
# K = 2 / 3 / 4
# =====================================================================

vote_z_thresholds = np.arange(
    0.5,
    6.01,
    0.1
)


abs_z_matrix = (
    valid_robust[
        ABS_Z_COLS
    ].values
)


for z_th in vote_z_thresholds:

    exceed_count = (
        abs_z_matrix
        >= z_th
    ).sum(
        axis=1
    )


    for k in [
        2,
        3,
        4
    ]:

        detector_pred = (
            exceed_count
            >= k
        ).astype(int)


        combined = (
            (base_pred == 1)
            |
            (detector_pred == 1)
        ).astype(int)


        all_rows.append({

            'method':
                f'VOTE_{k}of4',

            'threshold':
                z_th,

            'vote_k':
                k,

            'overall_f1':
                f1_score(
                    y_true,
                    combined,
                    zero_division=0
                ),

            'precision':
                precision_score(
                    y_true,
                    combined,
                    zero_division=0
                ),

            'recall':
                recall_score(
                    y_true,
                    combined,
                    zero_division=0
                ),

            'drift_recall':
                combined[
                    drift_mask
                ].mean(),

            'offset_recall':
                combined[
                    offset_mask
                ].mean(),

            'pred_ratio':
                combined.mean(),

            'detector_ratio':
                detector_pred.mean()
        })


# =====================================================================
# 7. 결과 DataFrame
# =====================================================================

aggregation_results = pd.DataFrame(
    all_rows
)


# =====================================================================
# 8. 전체 최고
# =====================================================================

best_idx = (
    aggregation_results[
        'overall_f1'
    ].idxmax()
)


best_row = (
    aggregation_results
    .loc[
        best_idx
    ]
)


print("\n" + "=" * 120)
print("전체 BEST")
print("=" * 120)

print(
    f"Method        = "
    f"{best_row['method']}"
)

print(
    f"Threshold     = "
    f"{best_row['threshold']:.2f}"
)

if pd.notna(
    best_row['vote_k']
):

    print(
        f"Vote K        = "
        f"{int(best_row['vote_k'])}"
    )


print(
    f"Combined F1   = "
    f"{best_row['overall_f1']:.6f}"
)

print(
    f"Delta F1      = "
    f"{best_row['overall_f1'] - BASE_F1:+.6f}"
)

print(
    f"Precision     = "
    f"{best_row['precision']:.6f}"
)

print(
    f"Recall        = "
    f"{best_row['recall']:.6f}"
)

print(
    f"Drift Recall  = "
    f"{best_row['drift_recall']:.6f}"
)

print(
    f"Offset Recall = "
    f"{best_row['offset_recall']:.6f}"
)

print(
    f"Pred Ratio    = "
    f"{best_row['pred_ratio']:.4%}"
)


# =====================================================================
# 9. Method별 최고 결과
# =====================================================================

best_by_method = (

    aggregation_results

    .sort_values(
        'overall_f1',
        ascending=False
    )

    .groupby(
        'method',
        as_index=False
    )

    .first()
)


print("\n" + "=" * 140)
print("METHOD별 최고 결과")
print("=" * 140)


print(
    best_by_method[
        [
            'method',
            'threshold',
            'vote_k',
            'overall_f1',
            'precision',
            'recall',
            'drift_recall',
            'offset_recall',
            'pred_ratio',
            'detector_ratio'
        ]
    ]
    .sort_values(
        'overall_f1',
        ascending=False
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 10. 전체 TOP 20
# =====================================================================

print("\n" + "=" * 150)
print("전체 TOP 20")
print("=" * 150)


print(
    aggregation_results
    .sort_values(
        'overall_f1',
        ascending=False
    )
    .head(
        20
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 11. Best 방식으로 prediction 생성
# =====================================================================

best_method = (
    best_row[
        'method'
    ]
)

best_th = (
    best_row[
        'threshold'
    ]
)


if best_method == 'MAX':

    detector_best = (
        valid_robust[
            'score_max'
        ].values
        >= best_th
    ).astype(int)


elif best_method == 'MEAN':

    detector_best = (
        valid_robust[
            'score_mean'
        ].values
        >= best_th
    ).astype(int)


elif best_method == 'MEDIAN':

    detector_best = (
        valid_robust[
            'score_median'
        ].values
        >= best_th
    ).astype(int)


else:

    k = int(
        best_row[
            'vote_k'
        ]
    )

    exceed_count = (
        abs_z_matrix
        >= best_th
    ).sum(
        axis=1
    )

    detector_best = (
        exceed_count
        >= k
    ).astype(int)


valid_robust[
    'best_robust_detector'
] = detector_best


valid_robust[
    'combined_robust_best'
] = (
    (
        valid_robust[
            'final_pred'
        ].values == 1
    )
    |
    (
        detector_best == 1
    )
).astype(int)


# =====================================================================
# 12. 문제 Case Recall 비교
# =====================================================================

problem_cases = [

    ('S-ORS', 2, 'drift'),

    ('S-ORS', 3, 'offset'),

    ('I-ORS', 1, 'drift'),

    ('I-ORS', 1, 'offset'),

    ('I-ORS', 1, 'drift+offset'),

    ('I-ORS', 7, 'drift'),

    ('I-ORS', 7, 'offset'),

    ('S-ORS', 8, 'drift'),

    ('S-ORS', 8, 'offset')
]


problem_rows = []


for station, layer, anomaly_type in problem_cases:

    mask = (
        (valid_robust['station'] == station)
        &
        (valid_robust['layer'] == layer)
        &
        (
            valid_robust[
                'anomaly_type'
            ] == anomaly_type
        )
    )


    temp = valid_robust.loc[
        mask
    ]


    if len(temp) == 0:

        continue


    problem_rows.append({

        'station':
            station,

        'layer':
            layer,

        'anomaly_type':
            anomaly_type,

        'N':
            len(temp),

        'base_recall':
            temp[
                'final_pred'
            ].mean(),

        'new_recall':
            temp[
                'combined_robust_best'
            ].mean(),

        'delta_recall':
            (
                temp[
                    'combined_robust_best'
                ].mean()
                -
                temp[
                    'final_pred'
                ].mean()
            ),

        'score_max_median':
            temp[
                'score_max'
            ].median(),

        'score_mean_median':
            temp[
                'score_mean'
            ].median(),

        'score_median_median':
            temp[
                'score_median'
            ].median()
    })


problem_aggregation_result = pd.DataFrame(
    problem_rows
)


print("\n" + "=" * 170)
print("문제 CASE : BEST AGGREGATION 결과")
print("=" * 170)


print(
    problem_aggregation_result
    .to_string(
        index=False
    )
)


# =====================================================================
# 13. Station별 비교
# =====================================================================

station_rows = []


for station in sorted(
    valid_robust[
        'station'
    ].unique()
):

    mask = (
        valid_robust[
            'station'
        ] == station
    )


    y_station = valid_robust.loc[
        mask,
        'label'
    ]


    base_station = valid_robust.loc[
        mask,
        'final_pred'
    ]


    new_station = valid_robust.loc[
        mask,
        'combined_robust_best'
    ]


    base_station_f1 = f1_score(
        y_station,
        base_station,
        zero_division=0
    )


    new_station_f1 = f1_score(
        y_station,
        new_station,
        zero_division=0
    )


    station_rows.append({

        'station':
            station,

        'base_f1':
            base_station_f1,

        'new_f1':
            new_station_f1,

        'delta_f1':
            new_station_f1
            -
            base_station_f1,

        'precision':
            precision_score(
                y_station,
                new_station,
                zero_division=0
            ),

        'recall':
            recall_score(
                y_station,
                new_station,
                zero_division=0
            )
    })


station_aggregation_result = pd.DataFrame(
    station_rows
)


print("\n" + "=" * 120)
print("STATION별 BASE VS BEST AGGREGATION")
print("=" * 120)


print(
    station_aggregation_result
    .to_string(
        index=False
    )
)



ROBUST AGGREGATION / VOTING TEST
BASE F1 = 0.634509

전체 BEST
Method        = MEAN
Threshold     = 10.00
Combined F1   = 0.633356
Delta F1      = -0.001153
Precision     = 0.673256
Recall        = 0.597922
Drift Recall  = 0.333239
Offset Recall = 0.457736
Pred Ratio    = 4.2298%

METHOD별 최고 결과
   method  threshold  vote_k  overall_f1  precision   recall  drift_recall  offset_recall  pred_ratio  detector_ratio
     MEAN       10.0     NaN    0.633356   0.673256 0.597922      0.333239       0.457736    0.042298        0.001422
   MEDIAN       10.0     NaN    0.633052   0.672568 0.597922      0.333239       0.457736    0.042342        0.001384
VOTE_4of4        4.6     4.0    0.631512   0.667341 0.599334      0.337183       0.457736    0.042774        0.003888
VOTE_3of4        6.0     3.0    0.623593   0.651567 0.597922      0.333239       0.457736    0.043706        0.004782
      MAX       10.0     NaN    0.609013   0.620524 0.597922      0.333239       0.457736    0.045893        0.00770

In [35]:
# =====================================================================
# STATION-ONLY THRESHOLD TEST
# Global vs Station vs Station×Layer
# =====================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 0. Speckle 함수
# =====================================================================

def apply_speckle_from_col(
    df,
    pred_col
):

    df_sorted = df.sort_values(
        'time'
    ).copy()


    s = df_sorted[
        pred_col
    ].astype(int).copy()


    is_one = pd.Series(
        s.values == 1,
        index=df_sorted.index
    )


    groups = pd.Series(
        s.values == 0,
        index=df_sorted.index
    ).cumsum()


    block_len = (
        is_one
        .groupby(groups)
        .transform('sum')
    )


    if 'abs_temp_diff_1' in df_sorted.columns:

        spike_in_group = (
            (
                df_sorted[
                    'abs_temp_diff_1'
                ] >= 1.7
            )
            .groupby(groups)
            .transform('any')
        )

    else:

        spike_in_group = pd.Series(
            False,
            index=df_sorted.index
        )


    remove_mask = (
        is_one
        &
        (block_len >= 2)
        &
        (block_len <= 11)
        &
        (~spike_in_group)
    )


    result = pd.Series(
        np.where(
            remove_mask,
            0,
            s.values
        ),
        index=df_sorted.index
    )


    return result.sort_index()


# =====================================================================
# 1. GLOBAL
# =====================================================================

global_rows = []


for th in np.arange(
    0.01,
    0.81,
    0.01
):

    pred = (
        valid['prob']
        >= th
    ).astype(int)


    global_rows.append({

        'threshold':
            th,

        'f1':
            f1_score(
                valid['label'],
                pred,
                zero_division=0
            )
    })


global_df = pd.DataFrame(
    global_rows
)


GLOBAL_TH = global_df.loc[
    global_df['f1'].idxmax(),
    'threshold'
]


valid[
    'pred_global'
] = (
    valid['prob']
    >= GLOBAL_TH
).astype(int)


# =====================================================================
# 2. STATION-ONLY
# =====================================================================

station_thresholds = {}


valid[
    'pred_station'
] = 0


for station in sorted(
    valid[
        'station'
    ].unique()
):

    mask = (
        valid[
            'station'
        ] == station
    )


    y_station = (
        valid.loc[
            mask,
            'label'
        ]
        .values
    )


    prob_station = (
        valid.loc[
            mask,
            'prob'
        ]
        .values
    )


    best_f1 = -1
    best_th = GLOBAL_TH


    for th in np.arange(
        0.01,
        0.81,
        0.01
    ):

        pred = (
            prob_station
            >= th
        ).astype(int)


        f1 = f1_score(
            y_station,
            pred,
            zero_division=0
        )


        if f1 > best_f1:

            best_f1 = f1
            best_th = th


    station_thresholds[
        station
    ] = best_th


    valid.loc[
        mask,
        'pred_station'
    ] = (
        valid.loc[
            mask,
            'prob'
        ]
        >= best_th
    ).astype(int)


    print(
        f"{station:6s} | "
        f"TH={best_th:.2f} | "
        f"F1={best_f1:.6f} | "
        f"N={mask.sum():,}"
    )


# =====================================================================
# 3. STATION × LAYER
#
# 기존 best_thresholds 사용
# =====================================================================

valid[
    'pred_station_layer'
] = 0


for station in valid[
    'station'
].unique():

    for layer in valid.loc[
        valid['station'] == station,
        'layer'
    ].unique():

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )


        th = best_thresholds[
            (station, layer)
        ]


        valid.loc[
            mask,
            'pred_station_layer'
        ] = (
            valid.loc[
                mask,
                'prob'
            ]
            >= th
        ).astype(int)


# =====================================================================
# 4. SPECKLE 적용
# =====================================================================

for pred_col in [
    'pred_global',
    'pred_station',
    'pred_station_layer'
]:

    final_col = (
        pred_col
        +
        '_speckle'
    )


    valid[
        final_col
    ] = 0


    for station in valid[
        'station'
    ].unique():

        for layer in valid.loc[
            valid['station'] == station,
            'layer'
        ].unique():

            mask = (
                (valid['station'] == station)
                &
                (valid['layer'] == layer)
            )


            valid.loc[
                mask,
                final_col
            ] = apply_speckle_from_col(

                valid.loc[
                    mask
                ],

                pred_col
            )


# =====================================================================
# 5. 전체 비교
# =====================================================================

methods = {

    'GLOBAL':
        'pred_global',

    'GLOBAL + Speckle':
        'pred_global_speckle',

    'STATION':
        'pred_station',

    'STATION + Speckle':
        'pred_station_speckle',

    'STATION_LAYER':
        'pred_station_layer',

    'STATION_LAYER + Speckle':
        'pred_station_layer_speckle'
}


rows = []


for name, col in methods.items():

    rows.append({

        'method':
            name,

        'f1':
            f1_score(
                valid['label'],
                valid[col],
                zero_division=0
            ),

        'precision':
            precision_score(
                valid['label'],
                valid[col],
                zero_division=0
            ),

        'recall':
            recall_score(
                valid['label'],
                valid[col],
                zero_division=0
            ),

        'pred_ratio':
            valid[col].mean()
    })


threshold_compare = pd.DataFrame(
    rows
)


print("\n" + "=" * 110)
print("GLOBAL vs STATION vs STATION × LAYER")
print("=" * 110)


print(
    threshold_compare
    .sort_values(
        'f1',
        ascending=False
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 6. Station별 결과
# =====================================================================

station_rows = []


for station in sorted(
    valid['station'].unique()
):

    mask = (
        valid['station']
        ==
        station
    )


    for name, col in methods.items():

        station_rows.append({

            'station':
                station,

            'method':
                name,

            'f1':
                f1_score(
                    valid.loc[
                        mask,
                        'label'
                    ],

                    valid.loc[
                        mask,
                        col
                    ],

                    zero_division=0
                )
        })


station_compare = pd.DataFrame(
    station_rows
)


print("\n" + "=" * 110)
print("STATION별 비교")
print("=" * 110)


print(
    station_compare
    .pivot(
        index='station',
        columns='method',
        values='f1'
    )
    .round(6)
    .to_string()
)


# =====================================================================
# 7. Threshold 출력
# =====================================================================

print("\n" + "=" * 80)
print("최종 Threshold")
print("=" * 80)


print(
    f"\nGLOBAL = "
    f"{GLOBAL_TH:.2f}"
)


print(
    "\nSTATION:"
)

for station, th in (
    station_thresholds.items()
):

    print(
        f"{station:6s} = "
        f"{th:.2f}"
    )


print(
    "\nSTATION × LAYER:"
)

for key in sorted(
    best_thresholds
):

    print(
        f"{key[0]:6s} "
        f"L{key[1]} = "
        f"{best_thresholds[key]:.2f}"
    )

G-ORS  | TH=0.17 | F1=0.246154 | N=16,930
I-ORS  | TH=0.17 | F1=0.488755 | N=64,521
S-ORS  | TH=0.34 | F1=0.655748 | N=126,642

GLOBAL vs STATION vs STATION × LAYER
                 method       f1  precision   recall  pred_ratio
STATION_LAYER + Speckle 0.634509   0.675867 0.597922    0.042135
          STATION_LAYER 0.627038   0.645813 0.609323    0.044937
      STATION + Speckle 0.582429   0.711519 0.492988    0.033000
                 GLOBAL 0.581212   0.720101 0.487236    0.032226
                STATION 0.580121   0.690337 0.500252    0.034513
       GLOBAL + Speckle 0.579056   0.731332 0.479265    0.031212

STATION별 비교
method     GLOBAL  GLOBAL + Speckle   STATION  STATION + Speckle  STATION_LAYER  STATION_LAYER + Speckle
station                                                                                                 
G-ORS    0.184438          0.195122  0.246154           0.255924       0.246154                 0.255924
I-ORS    0.482658          0.475082  0.488755       

In [37]:
# =====================================================================
# 35. STATION × LAYER THRESHOLD STABILITY TEST
#     2025 Jan-Mar vs Apr-Jun vs Full H1
# =====================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import f1_score


# =====================================================================
# 0. 기간 설정
# =====================================================================

valid = valid.copy()

valid['time'] = (
    pd.to_datetime(
        valid['time']
    )
    .dt.tz_localize(None)
)


periods = {

    'JAN_MAR': (
        pd.Timestamp('2025-01-01'),
        pd.Timestamp('2025-03-31 23:59:59')
    ),

    'APR_JUN': (
        pd.Timestamp('2025-04-01'),
        pd.Timestamp('2025-06-30 23:59:59')
    ),

    'FULL_H1': (
        pd.Timestamp('2025-01-01'),
        pd.Timestamp('2025-06-30 23:59:59')
    )
}


# =====================================================================
# 1. station × layer별 threshold 찾는 함수
# =====================================================================

def find_thresholds_by_period(
    df,
    start_date,
    end_date,
    threshold_grid=np.arange(
        0.01,
        0.81,
        0.01
    )
):

    period_df = df[
        (df['time'] >= start_date)
        &
        (df['time'] <= end_date)
    ].copy()


    results = []


    for station in sorted(
        period_df['station'].unique()
    ):

        layers = sorted(
            period_df.loc[
                period_df['station'] == station,
                'layer'
            ].unique()
        )


        for layer in layers:

            mask = (
                (period_df['station'] == station)
                &
                (period_df['layer'] == layer)
            )


            temp = period_df.loc[
                mask
            ].copy()


            n = len(temp)

            n_anomaly = int(
                temp['label'].sum()
            )


            # ---------------------------------------------------------
            # anomaly가 하나도 없으면 threshold 최적화 불가능
            # ---------------------------------------------------------

            if (
                n == 0
                or
                n_anomaly == 0
            ):

                results.append({

                    'station':
                        station,

                    'layer':
                        layer,

                    'N':
                        n,

                    'n_anomaly':
                        n_anomaly,

                    'anomaly_ratio':
                        (
                            n_anomaly / n
                            if n > 0
                            else np.nan
                        ),

                    'best_th':
                        np.nan,

                    'best_f1':
                        np.nan
                })

                continue


            y_true = (
                temp['label']
                .astype(int)
                .values
            )

            prob = (
                temp['prob']
                .values
            )


            best_f1 = -1
            best_th = np.nan


            for th in threshold_grid:

                pred = (
                    prob >= th
                ).astype(int)


                f1 = f1_score(
                    y_true,
                    pred,
                    zero_division=0
                )


                if f1 > best_f1:

                    best_f1 = f1
                    best_th = th


            results.append({

                'station':
                    station,

                'layer':
                    layer,

                'N':
                    n,

                'n_anomaly':
                    n_anomaly,

                'anomaly_ratio':
                    (
                        n_anomaly / n
                    ),

                'best_th':
                    best_th,

                'best_f1':
                    best_f1
            })


    return pd.DataFrame(
        results
    )


# =====================================================================
# 2. 세 기간 threshold 계산
# =====================================================================

period_results = {}


for period_name, (
    start_date,
    end_date
) in periods.items():

    result = find_thresholds_by_period(

        df=valid,

        start_date=start_date,

        end_date=end_date
    )


    period_results[
        period_name
    ] = result


    print(
        "\n" + "=" * 120
    )

    print(
        period_name
    )

    print(
        "=" * 120
    )


    print(
        result
        .to_string(
            index=False
        )
    )


# =====================================================================
# 3. 비교용 wide table
# =====================================================================

jan_mar = (
    period_results[
        'JAN_MAR'
    ]
    .rename(
        columns={
            'N':
                'N_JAN_MAR',

            'n_anomaly':
                'anom_JAN_MAR',

            'anomaly_ratio':
                'ratio_JAN_MAR',

            'best_th':
                'TH_JAN_MAR',

            'best_f1':
                'F1_JAN_MAR'
        }
    )
)


apr_jun = (
    period_results[
        'APR_JUN'
    ]
    .rename(
        columns={
            'N':
                'N_APR_JUN',

            'n_anomaly':
                'anom_APR_JUN',

            'anomaly_ratio':
                'ratio_APR_JUN',

            'best_th':
                'TH_APR_JUN',

            'best_f1':
                'F1_APR_JUN'
        }
    )
)


full_h1 = (
    period_results[
        'FULL_H1'
    ]
    .rename(
        columns={
            'N':
                'N_FULL',

            'n_anomaly':
                'anom_FULL',

            'anomaly_ratio':
                'ratio_FULL',

            'best_th':
                'TH_FULL',

            'best_f1':
                'F1_FULL'
        }
    )
)


threshold_stability = (

    full_h1

    .merge(
        jan_mar,
        on=[
            'station',
            'layer'
        ],
        how='outer'
    )

    .merge(
        apr_jun,
        on=[
            'station',
            'layer'
        ],
        how='outer'
    )
)


# =====================================================================
# 4. Threshold 차이 계산
# =====================================================================

threshold_stability[
    'TH_DIFF_HALF'
] = (
    threshold_stability[
        'TH_JAN_MAR'
    ]
    -
    threshold_stability[
        'TH_APR_JUN'
    ]
).abs()


threshold_stability[
    'TH_DIFF_JAN_FULL'
] = (
    threshold_stability[
        'TH_JAN_MAR'
    ]
    -
    threshold_stability[
        'TH_FULL'
    ]
).abs()


threshold_stability[
    'TH_DIFF_APR_FULL'
] = (
    threshold_stability[
        'TH_APR_JUN'
    ]
    -
    threshold_stability[
        'TH_FULL'
    ]
).abs()


# =====================================================================
# 5. 안정성 등급
#
# threshold 차이:
# <= 0.10  : Stable
# <= 0.25  : Moderate
# > 0.25   : Unstable
#
# 한쪽 기간 anomaly 0개면 Insufficient
# =====================================================================

def classify_stability(row):

    if (
        pd.isna(
            row[
                'TH_JAN_MAR'
            ]
        )
        or
        pd.isna(
            row[
                'TH_APR_JUN'
            ]
        )
    ):

        return 'INSUFFICIENT'


    diff = row[
        'TH_DIFF_HALF'
    ]


    if diff <= 0.10:

        return 'STABLE'


    elif diff <= 0.25:

        return 'MODERATE'


    else:

        return 'UNSTABLE'


threshold_stability[
    'stability'
] = (
    threshold_stability
    .apply(
        classify_stability,
        axis=1
    )
)


# =====================================================================
# 6. 안정 threshold 후보
#
# 두 half가 모두 존재하면 median
# 아니면 FULL threshold
# =====================================================================

def get_stable_threshold(row):

    values = []


    if pd.notna(
        row[
            'TH_JAN_MAR'
        ]
    ):

        values.append(
            row[
                'TH_JAN_MAR'
            ]
        )


    if pd.notna(
        row[
            'TH_APR_JUN'
        ]
    ):

        values.append(
            row[
                'TH_APR_JUN'
            ]
        )


    if len(values) >= 2:

        return float(
            np.median(
                values
            )
        )


    return row[
        'TH_FULL'
    ]


threshold_stability[
    'TH_MEDIAN_HALF'
] = (
    threshold_stability
    .apply(
        get_stable_threshold,
        axis=1
    )
)


# =====================================================================
# 7. 출력
# =====================================================================

show_cols = [

    'station',
    'layer',

    'anom_JAN_MAR',
    'anom_APR_JUN',
    'anom_FULL',

    'TH_JAN_MAR',
    'TH_APR_JUN',
    'TH_FULL',

    'TH_DIFF_HALF',

    'TH_MEDIAN_HALF',

    'stability'
]


print(
    "\n" + "=" * 150
)

print(
    "THRESHOLD STABILITY SUMMARY"
)

print(
    "=" * 150
)


print(
    threshold_stability[
        show_cols
    ]
    .sort_values(
        [
            'stability',
            'TH_DIFF_HALF'
        ],
        ascending=[
            True,
            False
        ]
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 8. Stability 개수
# =====================================================================

print(
    "\n" + "=" * 100
)

print(
    "STABILITY COUNTS"
)

print(
    "=" * 100
)


print(
    threshold_stability[
        'stability'
    ]
    .value_counts(
        dropna=False
    )
)


# =====================================================================
# 9. Threshold 차이가 큰 순서
# =====================================================================

print(
    "\n" + "=" * 140
)

print(
    "THRESHOLD 차이 큰 순서"
)

print(
    "=" * 140
)


print(
    threshold_stability[
        show_cols
    ]
    .sort_values(
        'TH_DIFF_HALF',
        ascending=False,
        na_position='last'
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 10. Median-half threshold를 validation 전체에 적용
#
# 성능 비교용
# =====================================================================

median_threshold_dict = {

    (
        row['station'],
        row['layer']
    ):
    row['TH_MEDIAN_HALF']

    for _, row in (
        threshold_stability
        .iterrows()
    )

    if pd.notna(
        row[
            'TH_MEDIAN_HALF'
        ]
    )
}


valid[
    'pred_median_half'
] = 0


for (
    station,
    layer
), th in (
    median_threshold_dict
    .items()
):

    mask = (
        (valid['station'] == station)
        &
        (valid['layer'] == layer)
    )


    valid.loc[
        mask,
        'pred_median_half'
    ] = (
        valid.loc[
            mask,
            'prob'
        ]
        >= th
    ).astype(int)


# =====================================================================
# 11. Speckle 적용
#
# 기존 apply_speckle_from_col 함수 사용
# =====================================================================

valid[
    'pred_median_half_speckle'
] = 0


for station in valid[
    'station'
].unique():

    for layer in valid.loc[
        valid['station'] == station,
        'layer'
    ].unique():

        mask = (
            (valid['station'] == station)
            &
            (valid['layer'] == layer)
        )


        valid.loc[
            mask,
            'pred_median_half_speckle'
        ] = apply_speckle_from_col(

            valid.loc[
                mask
            ],

            'pred_median_half'
        )


# =====================================================================
# 12. Full-H1 최적 threshold vs Median-half 비교
# =====================================================================

comparison_rows = []


for name, col in [

    (
        'FULL_H1 TH',
        'pred_station_layer'
    ),

    (
        'FULL_H1 TH + Speckle',
        'pred_station_layer_speckle'
    ),

    (
        'MEDIAN_HALF TH',
        'pred_median_half'
    ),

    (
        'MEDIAN_HALF TH + Speckle',
        'pred_median_half_speckle'
    )
]:

    comparison_rows.append({

        'method':
            name,

        'f1':
            f1_score(
                valid['label'],
                valid[col],
                zero_division=0
            )
    })


stability_performance = pd.DataFrame(
    comparison_rows
)


print(
    "\n" + "=" * 110
)

print(
    "FULL H1 THRESHOLD vs HALF-MEDIAN THRESHOLD"
)

print(
    "=" * 110
)


print(
    stability_performance
    .sort_values(
        'f1',
        ascending=False
    )
    .to_string(
        index=False
    )
)




JAN_MAR
station  layer     N  n_anomaly  anomaly_ratio  best_th  best_f1
  G-ORS      1 10187          1       0.000098     0.71 0.666667
  I-ORS      1 12906        201       0.015574     0.45 0.992519
  I-ORS      7 12906       1455       0.112738     0.01 0.508975
  S-ORS      1 12960        492       0.037963     0.03 0.460664
  S-ORS      5 12950       1174       0.090656     0.05 0.885009
  S-ORS      6    54          0       0.000000      NaN      NaN
  S-ORS      7    54          0       0.000000      NaN      NaN
  S-ORS      8 12906       1218       0.094375     0.01 0.501263

APR_JUN
station  layer     N  n_anomaly  anomaly_ratio  best_th  best_f1
  G-ORS      1  6743         71       0.010529     0.17 0.280612
  I-ORS      1  6321        772       0.122133     0.01 0.208850
  I-ORS      2  5802        301       0.051879     0.04 0.448833
  I-ORS      3  5804        257       0.044280     0.80 0.648101
  I-ORS      4  5801        228       0.039304     0.53 0.995595
  I-ORS

In [ ]:
# =====================================================================
# FINAL SUBMISSION A / B
#
# 모델 변경 없음
# - 기존 xgb_model 그대로 사용
# - 기존 150 all_features 그대로 사용
#
# CASE A
# Full-H1 Station × Layer Threshold + Speckle only
#
# CASE B
# Full-H1 Station × Layer Threshold
# 단, UNSTABLE / MODERATE layer만 Station Threshold로 fallback
# + Speckle only
# =====================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)


# =====================================================================
# 0. 현재 모델 상태 확인
# =====================================================================

assert 'xgb_model' in globals(), \
    "현재 학습된 xgb_model이 없습니다."

assert 'all_features' in globals(), \
    "all_features가 없습니다."

assert len(all_features) == 150, \
    f"현재 all_features가 {len(all_features)}개입니다. 150개여야 합니다."

assert 'best_thresholds' in globals(), \
    "best_thresholds가 없습니다."


# 우리가 확인한 Global threshold
GLOBAL_BEST_TH = 0.32


print("=" * 100)
print("FINAL SUBMISSION A / B")
print("=" * 100)

print(
    f"Base feature 수 = "
    f"{len(all_features)}"
)

print(
    f"Global fallback TH = "
    f"{GLOBAL_BEST_TH:.2f}"
)


# =====================================================================
# 1. CASE A Threshold
#
# 기존 Full-H1 Station × Layer
# =====================================================================

thresholds_A = (
    best_thresholds.copy()
)


print("\nCASE A Threshold")

for key in sorted(
    thresholds_A
):

    print(
        f"{key[0]:6s} "
        f"L{key[1]} = "
        f"{thresholds_A[key]:.2f}"
    )


# =====================================================================
# 2. CASE B Threshold
#
# 기본은 CASE A와 동일
#
# 안정성 검사에서
# UNSTABLE / MODERATE인 layer만
# Station threshold로 fallback
# =====================================================================

thresholds_B = (
    thresholds_A.copy()
)


# Validation에서 구했던 Station-only threshold
station_thresholds_final = {

    'G-ORS': 0.17,

    'I-ORS': 0.17,

    'S-ORS': 0.34
}


# -------------------------------------------------------------
# Stability 검사 결과
#
# UNSTABLE
# G-ORS L1
# I-ORS L1
# S-ORS L1
#
# MODERATE
# I-ORS L7
# S-ORS L5
# -------------------------------------------------------------

fallback_layers = [

    ('G-ORS', 1),

    ('I-ORS', 1),

    ('I-ORS', 7),

    ('S-ORS', 1),

    ('S-ORS', 5)
]


for station, layer in fallback_layers:

    thresholds_B[
        (station, layer)
    ] = (
        station_thresholds_final[
            station
        ]
    )


print("\n" + "=" * 100)
print("CASE B 변경 Threshold")
print("=" * 100)


for station, layer in fallback_layers:

    old_th = (
        thresholds_A[
            (station, layer)
        ]
    )

    new_th = (
        thresholds_B[
            (station, layer)
        ]
    )

    print(
        f"{station:6s} L{layer} : "
        f"{old_th:.2f} "
        f"→ "
        f"{new_th:.2f}"
    )


# =====================================================================
# 3. Speckle-only 함수
# =====================================================================

def apply_speckle_only_from_col(
    df,
    pred_col
):

    df_sorted = (
        df
        .sort_values(
            'time'
        )
        .copy()
    )


    s = (
        df_sorted[
            pred_col
        ]
        .astype(int)
        .copy()
    )


    # -------------------------------------------------------------
    # anomaly block
    # -------------------------------------------------------------

    is_one = pd.Series(

        s.values == 1,

        index=df_sorted.index
    )


    one_groups = pd.Series(

        s.values == 0,

        index=df_sorted.index

    ).cumsum()


    one_len = (
        is_one
        .groupby(
            one_groups
        )
        .transform(
            'sum'
        )
    )


    # -------------------------------------------------------------
    # Spike 보호
    # -------------------------------------------------------------

    if 'abs_temp_diff_1' in df_sorted.columns:

        spike_in_group = (

            (
                df_sorted[
                    'abs_temp_diff_1'
                ]
                >= 1.7
            )

            .groupby(
                one_groups
            )

            .transform(
                'any'
            )
        )

    else:

        spike_in_group = pd.Series(

            False,

            index=df_sorted.index
        )


    # -------------------------------------------------------------
    # 2~11 tick짜리 고립 anomaly 제거
    # 단 spike 보호
    # -------------------------------------------------------------

    remove_mask = (

        is_one

        &

        (one_len >= 2)

        &

        (one_len <= 11)

        &

        (~spike_in_group)
    )


    result = pd.Series(

        np.where(

            remove_mask,

            0,

            s.values
        ),

        index=df_sorted.index
    )


    return (
        result
        .sort_index()
        .astype(int)
    )


# =====================================================================
# 4. Threshold 적용 함수
# =====================================================================

def apply_threshold_dict(
    df,
    threshold_dict,
    prob_col='prob',
    output_col='base_pred'
):

    df[
        output_col
    ] = 0


    for station in df[
        'station'
    ].unique():

        layers = df.loc[

            df['station'] == station,

            'layer'

        ].unique()


        for layer in layers:

            mask = (

                (df['station'] == station)

                &

                (df['layer'] == layer)
            )


            th = threshold_dict.get(

                (station, layer),

                GLOBAL_BEST_TH
            )


            df.loc[
                mask,
                output_col
            ] = (

                df.loc[
                    mask,
                    prob_col
                ]

                >= th

            ).astype(int)


    df[
        output_col
    ] = (
        df[
            output_col
        ]
        .astype(int)
    )


    return df


# =====================================================================
# 5. Speckle 적용 함수
# =====================================================================

def apply_speckle_all_groups(
    df,
    pred_col,
    final_col
):

    df[
        final_col
    ] = 0


    for station in df[
        'station'
    ].unique():

        layers = df.loc[

            df['station'] == station,

            'layer'

        ].unique()


        for layer in layers:

            mask = (

                (df['station'] == station)

                &

                (df['layer'] == layer)
            )


            df.loc[
                mask,
                final_col
            ] = (
                apply_speckle_only_from_col(

                    df.loc[
                        mask
                    ],

                    pred_col
                )
            )


    df[
        final_col
    ] = (
        df[
            final_col
        ]
        .astype(int)
    )


    return df


# =====================================================================
# 6. 먼저 VALIDATION에서 A / B 비교
#
# 모델 재학습 없음
# 기존 valid['prob'] 그대로 사용
# =====================================================================

valid_submit_check = (
    valid.copy()
)


# CASE A
valid_submit_check = (
    apply_threshold_dict(

        valid_submit_check,

        thresholds_A,

        prob_col='prob',

        output_col='base_A'
    )
)


valid_submit_check = (
    apply_speckle_all_groups(

        valid_submit_check,

        pred_col='base_A',

        final_col='final_A'
    )
)


# CASE B
valid_submit_check = (
    apply_threshold_dict(

        valid_submit_check,

        thresholds_B,

        prob_col='prob',

        output_col='base_B'
    )
)


valid_submit_check = (
    apply_speckle_all_groups(

        valid_submit_check,

        pred_col='base_B',

        final_col='final_B'
    )
)


# =====================================================================
# 7. Validation 결과
# =====================================================================

validation_rows = []


for case_name, col in [

    (
        'CASE_A_FULL_LAYER',
        'final_A'
    ),

    (
        'CASE_B_STABILITY_FALLBACK',
        'final_B'
    )
]:

    validation_rows.append({

        'case':
            case_name,

        'f1':
            f1_score(

                valid_submit_check[
                    'label'
                ],

                valid_submit_check[
                    col
                ],

                zero_division=0
            ),

        'precision':
            precision_score(

                valid_submit_check[
                    'label'
                ],

                valid_submit_check[
                    col
                ],

                zero_division=0
            ),

        'recall':
            recall_score(

                valid_submit_check[
                    'label'
                ],

                valid_submit_check[
                    col
                ],

                zero_division=0
            ),

        'pred_ratio':
            valid_submit_check[
                col
            ].mean()
    })


validation_compare = pd.DataFrame(
    validation_rows
)


print("\n" + "=" * 110)
print("VALIDATION : SUBMISSION A vs B")
print("=" * 110)


print(
    validation_compare
    .to_string(
        index=False
    )
)


# =====================================================================
# 8. TEST / SAMPLE 로드
#
# 05_0903 파일과 동일한 제출 규격
# =====================================================================

test_file = (
    'test_features.csv'
)

sample_file = (
    'sample_submission.csv'
)


print("\nTest / Sample 로딩 중...")


test_2026 = pd.read_csv(

    test_file,

    low_memory=False
)


sample = pd.read_csv(

    sample_file
)


# =====================================================================
# 9. time 형식 통일
#
# Merge할 때 timezone 문제 방지
# =====================================================================

test_2026[
    'time'
] = pd.to_datetime(
    test_2026[
        'time'
    ],
    utc=True
)


sample[
    'time'
] = pd.to_datetime(
    sample[
        'time'
    ],
    utc=True
)


# =====================================================================
# 10. Test feature 확인
# =====================================================================

missing_features = [

    c
    for c in all_features

    if c not in test_2026.columns
]


if len(
    missing_features
) > 0:

    raise ValueError(

        "Test에 없는 feature가 있습니다:\n"

        +

        str(
            missing_features
        )
    )


X_test = (
    test_2026[
        all_features
    ]
)


print(
    f"Test Shape = "
    f"{test_2026.shape}"
)

print(
    f"Test Features = "
    f"{X_test.shape[1]}"
)


# =====================================================================
# 11. 현재 XGBoost로 Test probability
#
# ★ 모델 다시 학습하지 않음
# =====================================================================

print(
    "\n현재 xgb_model로 "
    "2026 Test probability 계산 중..."
)


test_2026[
    'prob'
] = (
    xgb_model
    .predict_proba(
        X_test
    )[:, 1]
)


print(
    test_2026[
        'prob'
    ].describe()
)


# =====================================================================
# 12. CASE A
#
# Full H1 Station × Layer
# + Speckle only
# =====================================================================

test_case_A = (
    test_2026.copy()
)


test_case_A = (
    apply_threshold_dict(

        test_case_A,

        thresholds_A,

        prob_col='prob',

        output_col='base_pred'
    )
)


test_case_A = (
    apply_speckle_all_groups(

        test_case_A,

        pred_col='base_pred',

        final_col='final_pred'
    )
)


# =====================================================================
# 13. CASE B
#
# Unstable / Moderate layer
# → Station threshold fallback
# + Speckle only
# =====================================================================

test_case_B = (
    test_2026.copy()
)


test_case_B = (
    apply_threshold_dict(

        test_case_B,

        thresholds_B,

        prob_col='prob',

        output_col='base_pred'
    )
)


test_case_B = (
    apply_speckle_all_groups(

        test_case_B,

        pred_col='base_pred',

        final_col='final_pred'
    )
)


# =====================================================================
# 14. Test prediction 비율 비교
# =====================================================================

print("\n" + "=" * 110)
print("TEST PREDICTION 비교")
print("=" * 110)


test_compare = pd.DataFrame({

    'case': [

        'CASE_A_FULL_LAYER',

        'CASE_B_STABILITY_FALLBACK'
    ],

    'anomaly_N': [

        int(
            test_case_A[
                'final_pred'
            ].sum()
        ),

        int(
            test_case_B[
                'final_pred'
            ].sum()
        )
    ],

    'anomaly_ratio': [

        test_case_A[
            'final_pred'
        ].mean(),

        test_case_B[
            'final_pred'
        ].mean()
    ]
})


print(
    test_compare
    .to_string(
        index=False
    )
)


# =====================================================================
# 15. A vs B 서로 다른 prediction 수
# =====================================================================

different_mask = (

    test_case_A[
        'final_pred'
    ].values

    !=

    test_case_B[
        'final_pred'
    ].values
)


print(
    "\nA/B Prediction 다른 행 = "
    f"{different_mask.sum():,}"
)

print(
    "전체 대비 = "
    f"{different_mask.mean():.4%}"
)


# =====================================================================
# 16. 제출파일 생성 함수
#
# 05_0903 notebook 규격:
#
# station
# year
# layer
# time
# label
# =====================================================================

key_cols = [

    'station',

    'year',

    'layer',

    'time'
]


def make_submission(
    test_df,
    sample_df,
    output_path
):

    pred_df = (
        test_df[
            key_cols
        ]
        .copy()
    )


    pred_df[
        'pred_label'
    ] = (

        test_df[
            'final_pred'
        ]
        .astype(int)
        .values
    )


    submission = (
        sample_df[
            key_cols
        ]
        .merge(

            pred_df,

            on=key_cols,

            how='left',

            validate='one_to_one'
        )
    )


    # -------------------------------------------------------------
    # Merge 누락 검사
    # -------------------------------------------------------------

    n_missing = (
        submission[
            'pred_label'
        ]
        .isna()
        .sum()
    )


    if n_missing > 0:

        raise ValueError(

            f"Submission merge 실패: "
            f"{n_missing:,}개 prediction 누락"
        )


    submission[
        'label'
    ] = (
        submission[
            'pred_label'
        ]
        .astype(int)
    )


    submission = submission[

        [
            'station',
            'year',
            'layer',
            'time',
            'label'
        ]
    ]


    # -------------------------------------------------------------
    # CSV의 time 문자열은 sample 원본 형식이 가장 안전
    #
    # sample을 다시 로드해서 time만 원래 문자열로 복원
    # -------------------------------------------------------------

    sample_original = pd.read_csv(
        sample_file
    )


    submission[
        'time'
    ] = (
        sample_original[
            'time'
        ].values
    )


    submission.to_csv(

        output_path,

        index=False
    )


    print(
        f"\nSAVED: "
        f"{output_path}"
    )

    print(
        f"Shape: "
        f"{submission.shape}"
    )

    print(
        f"Anomaly N: "
        f"{submission['label'].sum():,}"
    )

    print(
        f"Anomaly Ratio: "
        f"{submission['label'].mean():.4%}"
    )


    return submission


# =====================================================================
# 17. CASE A 저장
# =====================================================================

submission_A = (
    make_submission(

        test_case_A,

        sample,

        '../data/'
        'submission_CASE_A_fullH1_station_layer_speckle.csv'
    )
)


# =====================================================================
# 18. CASE B 저장
# =====================================================================

submission_B = (
    make_submission(

        test_case_B,

        sample,

        '../data/'
        'submission_CASE_B_stability_fallback_speckle.csv'
    )
)


# =====================================================================
# 19. 최종 sanity check
# =====================================================================

print("\n" + "=" * 110)
print("FINAL CHECK")
print("=" * 110)


print(
    "A shape:",
    submission_A.shape
)

print(
    "B shape:",
    submission_B.shape
)


print(
    "\nA label:"
)

print(
    submission_A[
        'label'
    ].value_counts()
)


print(
    "\nB label:"
)

print(
    submission_B[
        'label'
    ].value_counts()
)


print(
    "\nA/B label 다른 행:"
)

print(
    (
        submission_A[
            'label'
        ].values

        !=

        submission_B[
            'label'
        ].values
    ).sum()
)

FINAL SUBMISSION A / B
Base feature 수 = 150
Global fallback TH = 0.32

CASE A Threshold
G-ORS  L1 = 0.17
I-ORS  L1 = 0.74
I-ORS  L2 = 0.04
I-ORS  L3 = 0.80
I-ORS  L4 = 0.53
I-ORS  L5 = 0.05
I-ORS  L6 = 0.29
I-ORS  L7 = 0.02
S-ORS  L1 = 0.04
S-ORS  L2 = 0.79
S-ORS  L3 = 0.77
S-ORS  L4 = 0.65
S-ORS  L5 = 0.08
S-ORS  L6 = 0.21
S-ORS  L7 = 0.57
S-ORS  L8 = 0.01

CASE B 변경 Threshold
G-ORS  L1 : 0.17 → 0.17
I-ORS  L1 : 0.74 → 0.17
I-ORS  L7 : 0.02 → 0.17
S-ORS  L1 : 0.04 → 0.34
S-ORS  L5 : 0.08 → 0.34

VALIDATION : SUBMISSION A vs B
                     case       f1  precision   recall  pred_ratio
        CASE_A_FULL_LAYER 0.634509   0.675867 0.597922    0.042135
CASE_B_STABILITY_FALLBACK 0.616554   0.732805 0.532136    0.034585

Test / Sample 로딩 중...
Test Shape = (169011, 156)
Test Features = 150

현재 xgb_model로 2026 Test probability 계산 중...


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26164\1223924736.py:736: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_2026[


count    1.690110e+05
mean     3.933094e-02
std      1.621913e-01
min      1.765375e-08
25%      4.338352e-04
50%      1.379120e-03
75%      5.692333e-03
max      9.999791e-01
Name: prob, dtype: float64

TEST PREDICTION 비교
                     case  anomaly_N  anomaly_ratio
        CASE_A_FULL_LAYER       8574       0.050730
CASE_B_STABILITY_FALLBACK       7381       0.043672

A/B Prediction 다른 행 = 2,183
전체 대비 = 1.2916%

SAVED: ../data/submission_CASE_A_fullH1_station_layer_speckle.csv
Shape: (169011, 5)
Anomaly N: 8,574
Anomaly Ratio: 5.0730%

SAVED: ../data/submission_CASE_B_stability_fallback_speckle.csv
Shape: (169011, 5)
Anomaly N: 7,381
Anomaly Ratio: 4.3672%

FINAL CHECK
A shape: (169011, 5)
B shape: (169011, 5)

A label:
label
0    160437
1      8574
Name: count, dtype: int64

B label:
label
0    161630
1      7381
Name: count, dtype: int64

A/B label 다른 행:
2183


In [39]:
# =====================================================================
# CASE A vs CASE B
# TEST에서 실제로 어느 Station × Layer가 달라졌는지 분석
# =====================================================================

rows = []

for station in sorted(test_case_A['station'].unique()):

    layers = sorted(
        test_case_A.loc[
            test_case_A['station'] == station,
            'layer'
        ].unique()
    )

    for layer in layers:

        mask = (
            (test_case_A['station'] == station)
            &
            (test_case_A['layer'] == layer)
        )

        a = (
            test_case_A.loc[
                mask,
                'final_pred'
            ]
            .astype(int)
            .values
        )

        b = (
            test_case_B.loc[
                mask,
                'final_pred'
            ]
            .astype(int)
            .values
        )

        prob = (
            test_case_A.loc[
                mask,
                'prob'
            ]
            .values
        )

        # A=1 → B=0 : B에서 제거된 anomaly
        removed = (
            (a == 1)
            &
            (b == 0)
        )

        # A=0 → B=1 : B에서 새로 추가된 anomaly
        added = (
            (a == 0)
            &
            (b == 1)
        )

        diff = (
            a != b
        )

        rows.append({

            'station':
                station,

            'layer':
                layer,

            'N':
                mask.sum(),

            'A_anomaly':
                int(a.sum()),

            'B_anomaly':
                int(b.sum()),

            'A_ratio':
                a.mean(),

            'B_ratio':
                b.mean(),

            'A1_to_B0_removed':
                int(removed.sum()),

            'A0_to_B1_added':
                int(added.sum()),

            'total_diff':
                int(diff.sum()),

            'diff_ratio':
                diff.mean(),

            'prob_median':
                np.nanmedian(prob),

            'prob_q90':
                np.nanquantile(
                    prob,
                    0.90
                ),

            'prob_q95':
                np.nanquantile(
                    prob,
                    0.95
                ),

            'prob_q99':
                np.nanquantile(
                    prob,
                    0.99
                )
        })


ab_test_layer = pd.DataFrame(
    rows
)


print("\n" + "=" * 160)
print("TEST : CASE A vs CASE B — Station × Layer")
print("=" * 160)

print(
    ab_test_layer
    .sort_values(
        'total_diff',
        ascending=False
    )
    .to_string(
        index=False
    )
)


# =====================================================================
# 바뀐 layer만
# =====================================================================

changed_layers = ab_test_layer[
    ab_test_layer[
        'total_diff'
    ] > 0
].copy()


print("\n" + "=" * 160)
print("실제로 A/B prediction이 달라진 Layer만")
print("=" * 160)

print(
    changed_layers
    .sort_values(
        'total_diff',
        ascending=False
    )
    .to_string(
        index=False
    )
)


print("\nA → B에서 제거된 anomaly 총합:",
      changed_layers['A1_to_B0_removed'].sum())

print("A → B에서 추가된 anomaly 총합:",
      changed_layers['A0_to_B1_added'].sum())


# =====================================================================
# VALIDATION에서도 같은 layer가 어떻게 변했는지
# =====================================================================

val_rows = []

for station, layer in fallback_layers:

    mask = (
        (valid_submit_check['station'] == station)
        &
        (valid_submit_check['layer'] == layer)
    )

    temp = valid_submit_check.loc[
        mask
    ].copy()

    if len(temp) == 0:
        continue

    y = (
        temp['label']
        .astype(int)
        .values
    )

    a = (
        temp['final_A']
        .astype(int)
        .values
    )

    b = (
        temp['final_B']
        .astype(int)
        .values
    )

    val_rows.append({

        'station':
            station,

        'layer':
            layer,

        'N':
            len(temp),

        'true_anomaly':
            int(y.sum()),

        'A_pred':
            int(a.sum()),

        'B_pred':
            int(b.sum()),

        'A_F1':
            f1_score(
                y,
                a,
                zero_division=0
            ),

        'B_F1':
            f1_score(
                y,
                b,
                zero_division=0
            ),

        'A_precision':
            precision_score(
                y,
                a,
                zero_division=0
            ),

        'B_precision':
            precision_score(
                y,
                b,
                zero_division=0
            ),

        'A_recall':
            recall_score(
                y,
                a,
                zero_division=0
            ),

        'B_recall':
            recall_score(
                y,
                b,
                zero_division=0
            )
    })


ab_valid_layer = pd.DataFrame(
    val_rows
)


print("\n" + "=" * 160)
print("VALIDATION : A/B 변경 Layer 상세")
print("=" * 160)

print(
    ab_valid_layer
    .to_string(
        index=False
    )
)


TEST : CASE A vs CASE B — Station × Layer
station  layer     N  A_anomaly  B_anomaly  A_ratio  B_ratio  A1_to_B0_removed  A0_to_B1_added  total_diff  diff_ratio  prob_median  prob_q90  prob_q95  prob_q99
  I-ORS      7 19967       1220        479 0.061101 0.023990               743               2         745    0.037312     0.000839  0.011674  0.037379  0.999554
  S-ORS      1 15208       1475        784 0.096988 0.051552               691               0         691    0.045437     0.002101  0.052976  0.410669  0.999823
  I-ORS      1 16967        607       1100 0.035775 0.064832                 0             493         493    0.029056     0.002001  0.055546  0.397604  0.999070
  S-ORS      5 16795        976        722 0.058113 0.042989               254               0         254    0.015124     0.001287  0.020947  0.169280  0.999843
  I-ORS      4  6333        228        228 0.036002 0.036002                 0               0           0    0.000000     0.003871  0.075814  0.26

In [ ]:
# =====================================================================
# B ABLATION SUBMISSIONS
#
# 모델 변경 없음
# feature 변경 없음
# probability 재계산 없음
#
# B를 기준으로 변경 layer 하나씩만 A threshold로 원복
# =====================================================================

import pandas as pd
import numpy as np


# =====================================================================
# 1. Threshold 후보 만들기
# =====================================================================

thresholds_C = thresholds_B.copy()
thresholds_D = thresholds_B.copy()
thresholds_E = thresholds_B.copy()
thresholds_F = thresholds_B.copy()


# -------------------------------------------------------------
# C : I-ORS L7만 A로 원복
# B 0.17 → A 0.02
# -------------------------------------------------------------

thresholds_C[
    ('I-ORS', 7)
] = thresholds_A[
    ('I-ORS', 7)
]


# -------------------------------------------------------------
# D : S-ORS L1만 A로 원복
# B 0.34 → A 0.04
# -------------------------------------------------------------

thresholds_D[
    ('S-ORS', 1)
] = thresholds_A[
    ('S-ORS', 1)
]


# -------------------------------------------------------------
# E : I-ORS L1만 A로 원복
# B 0.17 → A 0.74
# -------------------------------------------------------------

thresholds_E[
    ('I-ORS', 1)
] = thresholds_A[
    ('I-ORS', 1)
]


# -------------------------------------------------------------
# F : S-ORS L5만 A로 원복
# B 0.34 → A 0.08
# -------------------------------------------------------------

thresholds_F[
    ('S-ORS', 5)
] = thresholds_A[
    ('S-ORS', 5)
]


# =====================================================================
# 2. 예측 생성 함수
# =====================================================================

def make_test_case(
    threshold_dict
):

    temp = test_2026.copy()


    temp = apply_threshold_dict(

        temp,

        threshold_dict,

        prob_col='prob',

        output_col='base_pred'
    )


    temp = apply_speckle_all_groups(

        temp,

        pred_col='base_pred',

        final_col='final_pred'
    )


    return temp


# =====================================================================
# 3. C / D / E / F 생성
# =====================================================================

test_case_C = make_test_case(
    thresholds_C
)

test_case_D = make_test_case(
    thresholds_D
)

test_case_E = make_test_case(
    thresholds_E
)

test_case_F = make_test_case(
    thresholds_F
)


# =====================================================================
# 4. B와 차이 확인
# =====================================================================

cases = {

    'B_BASELINE':
        test_case_B,

    'C_revert_I7':
        test_case_C,

    'D_revert_S1':
        test_case_D,

    'E_revert_I1':
        test_case_E,

    'F_revert_S5':
        test_case_F
}


rows = []


b_pred = (
    test_case_B[
        'final_pred'
    ]
    .astype(int)
    .values
)


for name, df in cases.items():

    pred = (
        df[
            'final_pred'
        ]
        .astype(int)
        .values
    )


    rows.append({

        'case':
            name,

        'anomaly_N':
            int(
                pred.sum()
            ),

        'anomaly_ratio':
            pred.mean(),

        'diff_vs_B':
            int(
                (
                    pred
                    !=
                    b_pred
                ).sum()
            )
    })


ablation_summary = pd.DataFrame(
    rows
)


print(
    "\n" + "=" * 100
)

print(
    "B ABLATION TEST SUMMARY"
)

print(
    "=" * 100
)


print(
    ablation_summary
    .to_string(
        index=False
    )
)


# =====================================================================
# 5. 제출 CSV 생성
# =====================================================================

submission_C = make_submission(

    test_case_C,

    sample,

    '../data/'
    'submission_CASE_C_B_revert_I7.csv'
)


submission_D = make_submission(

    test_case_D,

    sample,

    '../data/'
    'submission_CASE_D_B_revert_S1.csv'
)


submission_E = make_submission(

    test_case_E,

    sample,

    '../data/'
    'submission_CASE_E_B_revert_I1.csv'
)


submission_F = make_submission(

    test_case_F,

    sample,

    '../data/'
    'submission_CASE_F_B_revert_S5.csv'
)


print(
    "\n완료."
)

print(
    "B baseline Public F1 = 0.68989"
)

In [ ]:
# =====================================================================
# CASE G
# I-ORS는 B 유지
# S-ORS의 두 unstable layer는 validation 쪽으로 복원
# =====================================================================

thresholds_G = thresholds_B.copy()

# S-L1 : B 0.34 → 기존 Full-H1 0.04
thresholds_G[
    ('S-ORS', 1)
] = 0.04

# S-L5 : B 0.34 → 기존 Full-H1 0.08
thresholds_G[
    ('S-ORS', 5)
] = 0.08


# =====================================================================
# Test prediction
# =====================================================================

test_case_G = make_test_case(
    thresholds_G
)


# =====================================================================
# B / F / G 비교
# =====================================================================

for name, df in [
    ('B', test_case_B),
    ('F', test_case_F),
    ('G', test_case_G)
]:

    pred = df['final_pred'].astype(int)

    print(
        f"{name}: "
        f"N={pred.sum():,}, "
        f"ratio={pred.mean():.4%}"
    )


print(
    "\nG vs B 다른 행:",
    (
        test_case_G['final_pred'].values
        !=
        test_case_B['final_pred'].values
    ).sum()
)

print(
    "G vs F 다른 행:",
    (
        test_case_G['final_pred'].values
        !=
        test_case_F['final_pred'].values
    ).sum()
)


# =====================================================================
# Submission
# =====================================================================

submission_G = make_submission(

    test_case_G,

    sample,

    'submission.csv'
)

B: N=7,381, ratio=4.3672%
F: N=7,635, ratio=4.5175%
G: N=8,326, ratio=4.9263%

G vs B 다른 행: 945
G vs F 다른 행: 691

SAVED: ../data/submission_CASE_G_I_B_S_validation.csv
Shape: (169011, 5)
Anomaly N: 8,326
Anomaly Ratio: 4.9263%
